# Installation and Imports

In [1]:
!pip install -q PyPDF2 gdown ipywidgets pandas matplotlib nltk google-genai firebase-admin


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 25.7 MB/s eta 0:00:00


In [2]:
import os
import re
import glob
import html
import datetime
import requests
import gdown
import PyPDF2
import pandas as pd
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer
from nltk.chat.util import Chat, reflections
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, Output, ButtonStyle
import firebase_admin
from firebase_admin import credentials, db
from google import genai


# Configuration and Constants

In [3]:
# Google Drive folder with the PDF academic articles.
FOLDER_ID = "1zhHkG9eYIy6__Tnt7V-o_7293m3mz80W"
FOLDER_URL = f"https://drive.google.com/drive/folders/{FOLDER_ID}"

# Course IoT server:Only the humidity and temperature feeds work.
IOT_SERVER_BASE_URL = "https://server-cloud-v645.onrender.com"

USE_GEMINI = True
USE_FIREBASE = True

# Article metadata for the PDF files.
# This is a mapping of PDF filenames to their metadata (title, authors, URL).
# Key = PDF filename ("article1.pdf"). Missing entries fall back to the filename.
ARTICLE_INFO = {
    "bahmani-et-al-2015-silybum-marianum-beyond-hepatoprotection.pdf": {
        "title": "Silybum marianum: Beyond Hepatoprotection",
        "authors": (
            "Mahmood Bahmani, Hedayatollah Shirzad, "
            "Samira Rafieian, and Mahmoud Rafieian-Kopaei"
        ),
        "url": "https://doi.org/10.1177/2156587215571116"
    },

    "Annals of Applied Biology - 2015 - Andrzejewska - Silybum marianum  non‐medical exploitation of the species.pdf": {
        "title": "Silybum marianum: Non-Medical Exploitation of the Species",
        "authors": (
            "Jadwiga Andrzejewska, Tommaso Martinelli, "
            "and Katarzyna Sadowska"
        ),
        "url": "https://doi.org/10.1111/aab.12232"
    },

    "1-s2.0-S0926669013000034-main.pdf": {
        "title": (
            "Effects of Light Regimes on In Vitro Seed Germination "
            "and Silymarin Content in Silybum marianum"
        ),
        "authors": (
            "Mubarak Ali Khan, Bilal Haider Abbasi, "
            "Nisar Ahmed, and Huma Ali"
        ),
        "url": "https://doi.org/10.1016/j.indcrop.2012.12.035"
    },

    "Photochem   Photobiology - 2007 - Katiyar - Silymarin  a Flavonoid from Milk Thistle  Silybum marianum L    Inhibits.pdf": {
        "title": (
            "Silymarin, a Flavonoid from Milk Thistle "
            "(Silybum marianum L.), Inhibits UV-Induced Oxidative "
            "Stress Through Targeting Infiltrating CD11b+ Cells "
            "in Mouse Skin"
        ),
        "authors": (
            "Santosh K. Katiyar, Sreelatha Meleth, "
            "and Som D. Sharma"
        ),
        "url": "https://doi.org/10.1111/j.1751-1097.2007.00241.x"
    },

    "Annals of Applied Biology - 2014 - Martinelli - Phenological growth stages of Silybum marianum according to the extended.pdf": {
        "title": (
            "Phenological Growth Stages of Silybum marianum "
            "According to the Extended BBCH Scale"
        ),
        "authors": (
            "Tommaso Martinelli, Jadwiga Andrzejewska, "
            "M. Salis, and L. Sulas"
        ),
        "url": "https://doi.org/10.1111/aab.12163"
    },

    "molecules-22-01942.pdf": {
        "title": (
            "Silybin, a Major Bioactive Component of Milk Thistle "
            "(Silybum marianum L. Gaertn.)—Chemistry, "
            "Bioavailability, and Metabolism"
        ),
        "authors": "Michal Bijak",
        "url": "https://doi.org/10.3390/molecules22111942"
    },

    "agronomy-12-00729.pdf": {
        "title": (
            "Milk Thistle (Silybum marianum L.) as a Novel "
            "Multipurpose Crop for Agriculture in Marginal "
            "Environments: A Review"
        ),
        "authors": (
            "Roberto Marceddu, Lucia Dinolfo, Alessandra Carrubba, "
            "Mauro Sarno, and Giuseppe Di Miceli"
        ),
        "url": "https://doi.org/10.3390/agronomy12030729"
    }
}

# Stop words to exclude from the homework index table and search results.
STOP_WORDS = {
    "a", "an", "the", "and", "or", "but",
    "in", "on", "at", "of", "to", "for", "from",
    "with", "without", "by", "as", "into", "through",
    "during", "between", "among", "within", "over", "under",
    "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did",
    "can", "could", "may", "might", "must", "should", "would",
    "will", "shall",
    "this", "that", "these", "those",
    "it", "its", "they", "their", "them",
    "we", "our", "us", "you", "your",
    "he", "she", "his", "her",
    "which", "who", "whom", "whose", "what", "where",
    "when", "why", "how",
    "not", "no", "nor", "than", "then",
    "also", "very", "more", "most", "less",
    "such", "some", "any", "all", "each", "other",
    "both", "either", "several", "many", "much",
    "there", "here", "about", "because", "therefore",
    "however", "although", "while", "whereas"
}

# Additional domain-specific stop words commonly found in academic articles about plant biology and agriculture.
DOMAIN_STOP_WORDS = {
    "study", "studies", "research", "paper", "article",
    "review", "literature", "authors", "author",
    "result", "results", "finding", "findings",
    "method", "methods", "analysis", "data",
    "experiment", "experimental", "sample", "samples",
    "effect", "effects", "significant", "significantly",
    "observed", "reported", "shown", "showed",
    "investigated", "evaluated", "determined",
    "used", "using", "based", "including",
    "different", "various", "current", "previous",
    "present", "potential", "available",
    "conclusion", "conclusions", "objective", "aim",
    "introduction", "discussion", "figure", "table",
    "et", "al", "doi", "vol", "volume", "page", "pages"
}

STOP_WORDS.update(DOMAIN_STOP_WORDS)# Combine the general and domain-specific stop words into a single set for easy lookup.

# Academic terms used to build the homework index table (term | DocIDs).
IMPORTANT_TERMS = [
    # Plant identification
    "silybum", "marianum", "silymarin", "silybin", "thistle",
    # Plant biology and development
    "seed", "germination", "growth", "phenology", "morphology",
    "biomass", "fruit", "root", "shoot", "flowering",
    # Agricultural and environmental conditions
    "cultivation", "agronomy", "soil", "temperature", "humidity",
    "moisture", "light", "photoperiod", "irrigation", "drought",
    "stress", "environment",
    # Chemicals and biological components
    "flavonolignan", "flavonoid", "antioxidant", "enzyme",
    "extract", "oil", "protein",
    # Agricultural uses and plant health
    "crop", "yield", "medicinal", "bioenergy", "phytoremediation",
    "disease", "pathogen", "fungus", "infection"
]

# Porter Stemmer for normalizing terms in the homework index and search functionality.
stemmer = PorterStemmer()


# --------------- Plant profile and dashboard configuration ---------------

# Single configuration object for the monitored plant.
PLANT_PROFILE = {
    "plant_type":    "Milk Thistle",
    "location_type": "Indoor",       # "Indoor" or "Outdoor"
    "location_name": "Living Room",  # Room / area name; set to "" to hide
}

# Health thresholds used by calculate_plant_health_status().
HEALTH_THRESHOLDS = {
    "temperature": {
        "green_min":     15.0,  # °C — lower bound of preferred range
        "green_max":     28.0,  # °C — upper bound of preferred range
        "yellow_margin":  4.0,  # °C deviation → Yellow (Needs Attention)
        "orange_margin":  8.0,  # °C deviation → Orange (Warning); beyond = Red
    },
    "humidity": {
        "green_min":     40.0,  # % — lower bound of preferred range
        "green_max":     70.0,  # % — upper bound of preferred range
        "yellow_margin": 12.0,  # % deviation → Yellow
        "orange_margin": 22.0,  # % deviation → Orange; beyond = Red
    },
}

# Time window (hours) for the temperature fluctuation calculation on the dashboard.
TEMPERATURE_FLUCTUATION_HOURS = 6

# Minimum temperature difference (°C) required to classify a trend as Rising/Falling.
# Values within this delta from first→last reading are classified as Stable.
TEMPERATURE_STABLE_DELTA = 0.5


## Credentials

In [ ]:
# !!!!!Fill in your credentials below before running.!!!!!
# Do not share this file while credentials are present.!!!!

GEMINI_API_KEY = ""

FIREBASE_DATABASE_URL = "https://wombatcombat-7feee-default-rtdb.europe-west1.firebasedatabase.app/"

FIREBASE_SERVICE_ACCOUNT_JSON = {
   
}

if not GEMINI_API_KEY:
    USE_GEMINI = False
    print("GEMINI_API_KEY is empty. Gemini is disabled.")

if not FIREBASE_DATABASE_URL or not FIREBASE_SERVICE_ACCOUNT_JSON.get("private_key"):
    USE_FIREBASE = False
    print("Firebase credentials are incomplete. Firebase is disabled.")


# Shared Utilities

In [5]:
# Global variables to hold the state of the application and data:

FIREBASE_READY = False
FIREBASE_DB = None
GEMINI_CLIENT = None

ARTICLES = []              # loaded PDF documents
FULL_INVERTED_INDEX = {}   # all article vocabulary â€” used for search and RAG
HOMEWORK_INDEX = {}        # IMPORTANT_TERMS only â€” used for the homework table
STEMMED_TARGETS = {}       # stemmed form of IMPORTANT_TERMS

CHAT_HISTORY = []          # chatbot conversation for this session


## Text Processing Utilities

In [6]:
"""
Functions for text processing: tokenization, stop word removal, and stemming.
input: raw text string
output: list of processed tokens (lowercased, stop words removed, stemmed)
"""
def tokenize(text):
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())

"""
Function to preprocess text by tokenizing, removing stop words, and applying stemming.
input: raw text string
output: list of processed tokens (lowercased, stop words removed, stemmed)
"""
def preprocess_text(text):
    """Tokenize, remove stop words, and stem."""
    tokens = tokenize(text)
    return [stemmer.stem(t) for t in tokens if t not in STOP_WORDS]


# Search and RAG Service

## Article and PDF Loading

In [7]:

# --------------- PDF loading and text extraction functions ---------------

#path to local folder where PDFs will be stored after downloading from Google Drive.
PDF_FOLDER = "/content/drive_pdfs"
os.makedirs(PDF_FOLDER, exist_ok=True)

# Only download when no PDFs are present — avoids re-downloading on cell re-run.
existing_pdfs = glob.glob(os.path.join(PDF_FOLDER, "**", "*.pdf"), recursive=True)
if not existing_pdfs:
    print(f"Downloading PDFs from Google Drive: {FOLDER_URL}...")
    gdown.download_folder(url=FOLDER_URL, output=PDF_FOLDER, quiet=False, use_cookies=False)
    print("Download complete.\n")
else:
    print(f"PDFs already present ({len(existing_pdfs)} files). Skipping download.")

"""
Functions for loading PDF files, extracting text, and building the ARTICLES list.
input: folder path containing PDF files
output: list of article dicts with keys: id, title, authors, url, file
"""
def clean_filename_as_title(file_name):
    base = os.path.splitext(file_name)[0]
    return " ".join(base.replace("_", " ").replace("-", " ").split()).title()

"""
Function to extract text content from a PDF file.
input: path to PDF file
output: extracted text as a single string (empty if extraction fails)
"""
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + " "
    except Exception as e:
        print(f"Could not read {os.path.basename(pdf_path)}: {e}")
    return text.strip()

"""
Function to load articles from a folder containing PDF files.
input: folder path containing PDF files
output: list of article dicts with keys: id, title, authors, url, file
"""
def load_articles_from_folder(folder_path):
    """Load all PDFs from folder_path and return a list of article dicts."""
    pdf_files = sorted(glob.glob(os.path.join(folder_path, "**", "*.pdf"), recursive=True))
    articles = []

    if not pdf_files:
        print(f"No PDF files found in {folder_path}.")
        return articles

    for doc_id, path in enumerate(pdf_files, start=1):
        file_name = os.path.basename(path)
        text = extract_text_from_pdf(path)

        if not text:
            print(f"No readable text in {file_name} (may be image-based).")
            continue

        info = ARTICLE_INFO.get(file_name, {})
        articles.append({
            "id": f"doc{doc_id}",
            "title": info.get("title", clean_filename_as_title(file_name)),
            "authors": info.get("authors", "Metadata not configured"),
            "url": info.get("url", "Metadata not configured"),
            "file_name": file_name,
            "text": text
        })
        print(f"Loaded {file_name} as doc{doc_id}")

    print(f"\n{len(articles)} documents loaded.")
    return articles

# Load the articles from the PDF folder into the ARTICLES variable.
ARTICLES = load_articles_from_folder(PDF_FOLDER)


Retrieving folder contents


Processing file 171ODgBR_95AmCKdl8T2MAcU-xcfnSun5 1-s2.0-S0926669013000034-main.pdf
Processing file 1_i7YsU19bALkQiaGc-N3G_w28jp0ufUa agronomy-12-00729.pdf
Processing file 1wW45J8NXeTUXx_gow-AdgyDjuA193eRV Annals of Applied Biology - 2014 - Martinelli - Phenological growth stages of Silybum marianum according to the extended.pdf
Processing file 1W_ssooCnTVnb4oDLpMapP6w74UspF0eB Annals of Applied Biology - 2015 - Andrzejewska - Silybum marianum  non‐medical exploitation of the species.pdf
Processing file 10oSAqwvRkGZ28XaHdjYLUHjwMZ3hXU8J bahmani-et-al-2015-silybum-marianum-beyond-hepatoprotection.pdf
Processing file 1-wbn7RXKggd4LQHHeiMpQetCUYK464Q- molecules-22-01942.pdf
Processing file 1l6lDR08VJrWCFH-gElqCYfO_mgi8Em-a Photochem   Photobiology - 2007 - Katiyar - Silymarin  a Flavonoid from Milk Thistle  Silybum marianum L    Inhibits.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=171ODgBR_95AmCKdl8T2MAcU-xcfnSun5
To: /content/drive_pdfs/1-s2.0-S0926669013000034-main.pdf
100%|██████████| 575k/575k [00:00<00:00, 22.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1_i7YsU19bALkQiaGc-N3G_w28jp0ufUa
To: /content/drive_pdfs/agronomy-12-00729.pdf
100%|██████████| 1.64M/1.64M [00:00<00:00, 75.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wW45J8NXeTUXx_gow-AdgyDjuA193eRV
To: /content/drive_pdfs/Annals of Applied Biology - 2014 - Martinelli - Phenological growth stages of Silybum marianum according to the extended.pdf
100%|██████████| 2.36M/2.36M [00:00<00:00, 95.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1W_ssooCnTVnb4oDLpMapP6w74UspF0eB
To: /content/drive_pdfs/Annals of Applied Biology - 2015 - Andrzejewska - Silybum marianum  non‐medical exploitation of the species.pdf
100%|

Download complete.

Loaded 1-s2.0-S0926669013000034-main.pdf as doc1
Loaded Annals of Applied Biology - 2014 - Martinelli - Phenological growth stages of Silybum marianum according to the extended.pdf as doc2
Loaded Annals of Applied Biology - 2015 - Andrzejewska - Silybum marianum  non‐medical exploitation of the species.pdf as doc3
Loaded Photochem   Photobiology - 2007 - Katiyar - Silymarin  a Flavonoid from Milk Thistle  Silybum marianum L    Inhibits.pdf as doc4
Loaded agronomy-12-00729.pdf as doc5
Loaded bahmani-et-al-2015-silybum-marianum-beyond-hepatoprotection.pdf as doc6
Loaded molecules-22-01942.pdf as doc7

7 documents loaded.


## Search Index Service

In [8]:
#--------------- Index building and search functions ---------------

"""
Functions for building the full inverted index, the homework index, and searching articles.
input: ARTICLES list and IMPORTANT_TERMS list
output: FULL_INVERTED_INDEX dict, HOMEWORK_INDEX dict, STEMMED_TARGETS dict
"""
def build_full_inverted_index(articles):
    index = {}
    for article in articles:
        doc_id = article["id"]
        for term in set(preprocess_text(article["text"])):
            index.setdefault(term, set()).add(doc_id)
    return index

"""
Function to build an index limited to IMPORTANT_TERMS only. Used for the homework term | DocIDs table.
input: ARTICLES list and IMPORTANT_TERMS list
output: HOMEWORK_INDEX dict (stemmed term → set of DocIDs), STEMMED_TARGETS dict
"""
def build_homework_index(articles, important_terms):
    stemmed_targets = {term: stemmer.stem(term.lower()) for term in important_terms}
    index = {s: set() for s in stemmed_targets.values()}

    for article in articles:
        doc_id = article["id"]
        terms_in_doc = set(preprocess_text(article["text"]))
        for _, stemmed in stemmed_targets.items():
            if stemmed in terms_in_doc:
                index[stemmed].add(doc_id)

    return index, stemmed_targets

"""
Function to create the homework term | DocIDs table as a DataFrame.
input: HOMEWORK_INDEX dict (stemmed term → set of DocIDs), STEMMED_TARGETS dict
output: DataFrame with columns "term" (original term) and "DocIDs" (comma-separated list of DocIDs or "None")
"""
def build_index_dataframe(homework_index, stemmed_targets):
    rows = []
    for original, stemmed in stemmed_targets.items():
        doc_ids = sorted(homework_index.get(stemmed, set()))
        rows.append({
            "term": original,
            "DocIDs": ", ".join(doc_ids) if doc_ids else "None"
        })
    return pd.DataFrame(rows).sort_values("term").reset_index(drop=True)

"""
Function to search articles using the inverted index.
input: query (string), articles (list of dicts), inverted_index (dict)
output: list of tuples (article, score, matched_terms)
"""
def search_articles(query, articles, inverted_index):
    query_terms = preprocess_text(query)
    scores = {}
    matched_by_doc = {}

    for term in query_terms:
        if term in inverted_index:
            for doc_id in inverted_index[term]:
                scores[doc_id] = scores.get(doc_id, 0) + 1
                matched_by_doc.setdefault(doc_id, set()).add(term)

    sorted_ids = sorted(scores, key=lambda d: scores[d], reverse=True)
    results = []
    for doc_id in sorted_ids:
        article = next((a for a in articles if a["id"] == doc_id), None)
        if article:
            results.append((article, scores[doc_id], sorted(matched_by_doc.get(doc_id, []))))

    return results, query_terms

"""
Function to build a text context block from retrieved articles for the RAG prompt.
input: list of tuples (article, score, matched_terms), max_chars_per_doc (int)
output: formatted string with article excerpts and metadata for RAG context
"""
def build_context_from_results(results, max_chars_per_doc=1200):
    if not results:
        return ""
    parts = []
    for rank, (article, score, matched_terms) in enumerate(results, start=1):
        parts.append(
            f"Source {rank}: {article['title']} ({article['id']})\n"
            f"Matched terms: {', '.join(matched_terms)}\n"
            f"Text excerpt:\n{article['text'][:max_chars_per_doc]}\n"
        )
    return "\n---\n".join(parts)

# Build the full inverted index and the homework index from the loaded ARTICLES.
FULL_INVERTED_INDEX = build_full_inverted_index(ARTICLES)
HOMEWORK_INDEX, STEMMED_TARGETS = build_homework_index(ARTICLES, IMPORTANT_TERMS)
index_df = build_index_dataframe(HOMEWORK_INDEX, STEMMED_TARGETS)


## RAG Search Service

In [9]:
#--------------- RAG answer generation functions ---------------

"""
Functions to generate answers using retrieved articles, with a simple local fallback when Gemini is unavailable.
input: query (string), results (list of tuples (article, score, matched_terms))
output: answer string to be displayed on the dashboard
"""
def simple_rag_answer(query, results):
    if not results:
        return (
            "No relevant academic article was found for this query. "
            "Try searching for terms such as mildew, humidity, moisture, leaf, disease, or temperature."
        )

    top_article, top_score, matched_terms = results[0]
    matched_text = ", ".join(matched_terms) if matched_terms else "the query terms"

    return (
        f"The query is most related to <b>{html.escape(top_article['title'])}</b>. "
        f"The system matched the academic terms: <b>{html.escape(matched_text)}</b>. "
        f"Based on the retrieved context, the Milk Thistle plant should be checked for visible leaf symptoms, "
        f"humidity, temperature, and general disease indicators. "
        f"For a real recommendation, combine the retrieved article context with the current IoT readings."
    )

"""
Function to generate an answer using Gemini with the retrieved article context. If Gemini is unavailable, it falls back to the simple_rag_answer.
input: query (string), results (list of tuples (article, score, matched_terms))
output: answer string generated by Gemini or the fallback method
"""
def gemini_rag_answer(query, results):
    if GEMINI_CLIENT is None:
        return simple_rag_answer(query, results)

    context = build_context_from_results(results)
    prompt = f"""
    You are an academic assistant inside a cloud computing RAG prototype for Milk Thistle plant monitoring.

    Your role:
    The user asks a question about Milk Thistle plants, diseases, humidity, water stress, temperature,
    growth conditions, or plant health.
    You must answer using ONLY the retrieved academic context below.
    Do NOT use outside knowledge.
    Do NOT invent facts that are not supported by the retrieved context.

    Important behavior:
    - If the retrieved context directly answers the question, give a clear academic answer.
    - If the retrieved context is only partially related, clearly say:
      "The retrieved articles provide related evidence, but not a full direct answer."
    - If the user asks about mildew but the context mentions other fungal diseases, explain that the articles discuss fungal disease risk generally, but may not directly discuss mildew.
    - Mention specific diseases, symptoms, or environmental conditions only if they appear in the retrieved context.
    - Keep the answer useful for a Milk Thistle monitoring dashboard.

    Required answer format:
    1. Direct answer: 1-2 sentences.
    2. Evidence from retrieved articles: 2-3 sentences.
    3. Monitoring implication: 1 sentence.
    4. Limitation: 1 sentence if the context is partial.

    Academic context:
    {context}

    User question:
    {query}
    """

    try:
        response = GEMINI_CLIENT.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        return response.text

    except Exception as api_error:
        error_str = str(api_error).lower()
        fallback_text = simple_rag_answer(query, results)
        print(f"Gemini RAG error: {api_error}")

        if "429" in error_str or "quota" in error_str or "exhausted" in error_str:
            return f"Gemini is temporarily unavailable, so the local fallback was used.\n\n{fallback_text}"
        elif "503" in error_str or "unavailable" in error_str:
            return f"Gemini is temporarily unavailable, so the local fallback was used.\n\n{fallback_text}"
        else:
            return f"Gemini is temporarily unavailable, so the local fallback was used.\n\n{fallback_text}"

"""
Function to generate an answer for the dashboard using RAG. It tries to use Gemini if available, and falls back to a simple local answer if Gemini is unavailable or if an error occurs.
input: query (string), results (list of tuples (article, score, matched_terms))
output: answer string to be displayed on the dashboard, generated by Gemini or the fallback method
"""
def rag_answer(query, results):
    if USE_GEMINI:
        return gemini_rag_answer(query, results)
    return simple_rag_answer(query, results)


# Firebase Service

In [10]:
#---------------- Firebase integration functions ---------------

"""
Function to initialize the Firebase connection using the provided credentials. Sets global variables for the database reference and connection status.
input: none (uses global credentials)
output: success status (True/False) of the initialization process
"""
def initialize_firebase():
    global FIREBASE_READY, FIREBASE_DB

    if not FIREBASE_DATABASE_URL or not FIREBASE_SERVICE_ACCOUNT_JSON.get("private_key"):
        print("Firebase credentials missing.")
        return False

    try:
        if not firebase_admin._apps:
            cred = credentials.Certificate(FIREBASE_SERVICE_ACCOUNT_JSON)
            firebase_admin.initialize_app(cred, {"databaseURL": FIREBASE_DATABASE_URL})

        FIREBASE_DB = db
        FIREBASE_READY = True
        print("Firebase connected successfully")
        return True

    except Exception as e:
        FIREBASE_READY = False
        print(f"Firebase initialization failed: {e}")
        return False


"""
Function to save the homework index DataFrame to Firebase. Converts the DataFrame to a list of dicts for storage.
input: index_rows (DataFrame containing the homework index)
output: success status (True/False) of the save operation
"""
def firebase_save_index(index_rows):
    if not FIREBASE_READY:
        return False
    try:
        FIREBASE_DB.reference("index").set(index_rows.to_dict(orient="records"))
        return True
    except Exception as e:
        print(f"Could not save index to Firebase: {e}")
        return False

"""
Function to save article metadata (title, authors, URL, filename) to Firebase.
input: articles (list of dicts containing article metadata)
output: success status (True/False) of the save operation
"""
def firebase_save_articles_metadata(articles):
    if not FIREBASE_READY:
        return False
    try:
        metadata = {
            a["id"]: {
                "title": a.get("title", ""),
                "authors": a.get("authors", ""),
                "url": a.get("url", ""),
                "file_name": a.get("file_name", "")
            }
            for a in articles
        }
        FIREBASE_DB.reference("articles").set(metadata)
        return True
    except Exception as e:
        print(f"Could not save article metadata to Firebase: {e}")
        return False

"""
Function to save sensor history data to Firebase. Converts the DataFrame to a list of dicts and formats timestamps as ISO strings for storage.
input: feed (string identifier for the sensor feed), df (DataFrame containing sensor samples with a "created_at" timestamp column)
output: success status (True/False) of the save operation
"""
def firebase_save_sensor_history(feed, df):
    if not FIREBASE_READY or df is None or df.empty:
        return False

    try:
        # Work on a copy so the original DataFrame is not changed.
        firebase_df = df.copy()

        # Firebase accepts strings, but not pandas Timestamp objects.
        if "created_at" in firebase_df.columns:
            firebase_df["created_at"] = firebase_df["created_at"].apply(
                lambda value: value.isoformat()
                if pd.notna(value) and hasattr(value, "isoformat")
                else str(value)
            )

        records = firebase_df.to_dict(orient="records")

        payload = {
            "feed": feed,
            "samples_count": len(records),
            "created_at": datetime.datetime.now(
                datetime.timezone.utc
            ).isoformat(),
            "samples": records
        }

        FIREBASE_DB.reference(
            f"sensor_readings/{feed}"
        ).push(payload)

        return True

    except Exception as e:
        print(f"Could not save {feed} sensor history to Firebase: {e}")
        return False

"""
Function to save the latest sensor summary (current readings and health status) to Firebase. This is used for the dashboard and chatbot to read the most recent plant status.
This is used by the dashboard and chatbot to read the most recent plant status.
The summary dict should contain the latest sensor readings and a calculated health status (e.g., "Good", "Needs Attention", "Warning").
input: summary (dict containing the latest sensor readings and health status)
output: success status (True/False) of the save operation
"""
def firebase_save_latest_sensor_summary(summary):
    if not FIREBASE_READY:
        return False
    try:
        FIREBASE_DB.reference("latest_sensor_reading").set(summary)
        return True
    except Exception as e:
        print(f"Could not save sensor summary to Firebase: {e}")
        return False

"""
Function to read the latest sensor summary from Firebase.
This is used by the dashboard and chatbot to get the most recent plant status for answering questions and displaying on the dashboard.
input: none
output: dict containing the latest sensor readings and health status, or None if reading fails
"""
def firebase_get_latest_sensor_reading():
    if not FIREBASE_READY:
        return None
    try:
        return FIREBASE_DB.reference("latest_sensor_reading").get()
    except Exception as e:
        print(f"Could not read sensor data from Firebase: {e}")
        return None

"""
Function to add a journal event to Firebase. The payload should be a dict containing the event details (e.g., type, description, timestamp).
This allows the user to log manual observations or events related to the plant, which can be displayed
on the dashboard and used for tracking plant health over time.
input: payload (dict containing event details such as type, description, timestamp)
output: dict with keys {ok: bool, key: str} if successful, or {ok: bool, error: str} if an error occurs
"""
def firebase_add_journal_event(payload):
    if not FIREBASE_READY:
        return {"ok": False, "error": "Firebase is not connected."}
    try:
        ref = FIREBASE_DB.reference("plant_journal").push(payload)
        return {"ok": True, "key": ref.key}
    except Exception as e:
        return {"ok": False, "error": str(e)}

"""
Function to read all journal events from Firebase. This retrieves the raw event records that have been logged, which can be displayed on the dashboard or used for analysis.
input: none
output: list of dicts representing journal events, or an empty list if reading fails
"""
def firebase_get_journal_events():
    if not FIREBASE_READY:
        return []
    try:
        data = FIREBASE_DB.reference("plant_journal").get()
        if not data:
            return []
        events = []
        for key, val in data.items():
            if not isinstance(val, dict):
                continue
            entry = dict(val)
            entry["_key"] = key
            events.append(entry)
        return events
    except Exception as e:
        print(f"Could not read journal events from Firebase: {e}")
        return []

"""
Function to delete a specific journal event from Firebase using its unique Firebase key.
The event key is created automatically when the event is added to the "plant_journal" collection.
input: event_key (string containing the unique Firebase key of the journal event)
output: dict with keys {ok: bool} if the event is deleted successfully,
or {ok: bool, error: str} if Firebase is unavailable, the key is missing, or an error occurs
"""
def firebase_delete_journal_event(event_key):
    if not FIREBASE_READY:
        return {
            "ok": False,
            "error": "Firebase is not connected."
        }

    if not event_key:
        return {
            "ok": False,
            "error": "Journal event key is missing."
        }

    try:
        FIREBASE_DB.reference(
            f"plant_journal/{event_key}"
        ).delete()

        return {
            "ok": True
        }

    except Exception as e:
        return {
            "ok": False,
            "error": str(e)
        }


#------------------- Initialization and data saving to Firebase ---------------
if USE_FIREBASE:
    initialize_firebase()
else:
    print("Firebase is disabled.")

firebase_save_index(index_df)
firebase_save_articles_metadata(ARTICLES)


Firebase connected successfully


True

# Sensor Service

In [11]:
"""
Function to fetch sensor data from the course IoT server for a specified feed (e.g., humidity, temperature).
The function makes an HTTP GET request to the server's history endpoint, retrieves the data, and returns it as a DataFrame.
If the request fails or the response is invalid, it returns None and an error message.
input: feed (string identifier for the sensor feed, e.g., "humidity"), limit (int number of recent samples to fetch)
output: tuple (DataFrame containing the sensor data, or None if fetching fails; dict with error message if an error occurs, or None if successful)
"""
def fetch_iot_feed(feed="humidity", limit=10):
    try:
        response = requests.get(
            f"{IOT_SERVER_BASE_URL}/history",
            params={"feed": feed, "limit": limit},
            timeout=90
        )
        response.raise_for_status()
        data = response.json()

        if "data" not in data:
            return None, data

        df = pd.DataFrame(data["data"])

        if "created_at" in df.columns:
            df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
        if "value" in df.columns:
            df["value"] = pd.to_numeric(df["value"], errors="coerce")

        return df, None

    except requests.RequestException as e:
        print(f"Sensor request failed ({feed}): {e}")
        return None, {"error": str(e)}
    except Exception as e:
        return None, {"error": str(e)}

"""
Function to extract the latest sensor value from a DataFrame returned by fetch_iot_feed().
input: df (DataFrame containing sensor data with a "value" column)
output: the latest sensor value (first row's "value") or None if the DataFrame is empty or invalid
"""
def extract_latest_feed_value(df):
    if df is None or df.empty:
        return None
    return df.iloc[0].get("value")


# Gemini Service

In [12]:
"""
Function to initialize the Gemini client using the provided API key. Sets the global GEMINI_CLIENT variable if successful.
input: none (uses global GEMINI_API_KEY)
output: success status (True/False) of the initialization process
"""
def initialize_gemini():
    global GEMINI_CLIENT

    if not GEMINI_API_KEY:
        print("Gemini API key is empty.")
        return False

    try:
        GEMINI_CLIENT = genai.Client(api_key=GEMINI_API_KEY)
        print("Gemini connected successfully")
        return True
    except Exception as e:
        print(f"Gemini initialization failed: {e}")
        return False


if USE_GEMINI:
    initialize_gemini()


Gemini connected successfully


# Chatbot Service

In [13]:
# pattern list for the rule-based chatbot fallback.
# Each pattern is a regex that matches user messages, and a list of possible responses.
# The chatbot will check the patterns in order and respond with the first matching pattern's response.
# If no patterns match, it will use a default response.
CHATBOT_PATTERNS = [
    [
        r".*(hello|hi|hey).*",
        ["Hello! I am the WombatCombat Plant Care chatbot. You can ask me about Milk Thistle care, humidity, diseases, sensors, or the academic articles."]
    ],
    [
        r".*(humidity|moisture|water|irrigation).*",
        ["Humidity and moisture are important for Milk Thistle monitoring. You can check the sensor screen and also search the academic articles for humidity, moisture, water, or irrigation."]
    ],
    [
        r".*(disease|mildew|fungus|fungal|yellow|spots).*",
        ["For possible plant disease, check visible symptoms such as yellowing, spots, or leaf damage. The academic search can retrieve article context about disease, mildew, fungus, and symptoms."]
    ],
    [
        r".*(sensor|iot|temperature).*",
        ["The IoT sensors collect temperature and air humidity readings. These values are shown on the Sensors and Dashboard screens."]
    ],
    [
        r".*(what can you do|help|options).*",
        ["I can answer questions about Milk Thistle care, explain sensor readings, connect your question to the academic RAG search, and give practical monitoring suggestions."]
    ],
    [
        r".*",
        ["I am not sure based on the rule-based chatbot. I will try to answer using Gemini if it is connected."]
    ]
]

RULE_BASED_CHATBOT = Chat(CHATBOT_PATTERNS, reflections) # simple pattern-matching chatbot for fallback when Gemini is unavailable

"""
Function to build a short text summary of the latest sensor reading from Firebase.
This summary is used as part of the context for the chatbot to answer user questions about the current plant status.
If no sensor readings are available, it returns a message indicating that the user should update the Sensors screen first.
input: none (reads the latest sensor reading from Firebase)
output: string summary of the latest sensor reading, or a message indicating no readings are available
"""
def build_sensor_context_for_chatbot():
    reading = firebase_get_latest_sensor_reading()
    if not reading:
        return "No sensor readings are available. Update the Sensors screen first."
    return (
        f"Latest sensor reading — "
        f"Temperature: {reading.get('temperature', 'unknown')}, "
        f"Air humidity: {reading.get('air_humidity', 'unknown')}, "
        f"Updated: {reading.get('updated_at', 'unknown')}"
    )

"""
Function to generate a chatbot answer using Gemini with RAG context and sensor data.
If Gemini is unavailable or an error occurs, it falls back to a simple rule-based chatbot response.
input: user_message (string containing the user's question or message)
output: string answer generated by Gemini or the fallback chatbot
"""
def chatbot_answer(user_message):
    user_message = user_message.strip()
    if not user_message:
        return "Please type a question first."

    results, _ = search_articles(user_message, ARTICLES, FULL_INVERTED_INDEX)
    academic_context = build_context_from_results(results, max_chars_per_doc=900)
    sensor_context = build_sensor_context_for_chatbot()

    if USE_GEMINI and GEMINI_CLIENT is not None:
        try:
            prompt = f"""
            You are an AI chatbot inside a Google Colab cloud prototype called WombatCombat Plant Care.

            The system monitors Milk Thistle plants using:
            - academic article search / RAG
            - Firebase data
            - IoT sensor readings (temperature and air humidity)
            - image upload screen (not yet connected to a model)

            Answer the user in a helpful and simple way.

            Rules:
            - Keep the answer practical and understandable.
            - Use the academic context when available.
            - If sensor data is missing, say so.
            - Do not invent scientific facts not found in the context.
            - Use 3-6 short sentences.

            Sensor context:
            {sensor_context}

            Academic context:
            {academic_context if academic_context else "No relevant academic context was retrieved."}

            User question:
            {user_message}
            """

            response = GEMINI_CLIENT.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            return response.text

        except Exception as e:
            print(f"Gemini chatbot error: {e}")
            fallback = RULE_BASED_CHATBOT.respond(user_message) or "I could not find a standard answer."
            return f"{fallback}\n\nGemini is temporarily unavailable, so the local fallback was used."

    fallback = RULE_BASED_CHATBOT.respond(user_message)
    return fallback or "I could not generate an answer. Try asking about Milk Thistle, humidity, moisture, disease, sensors, or temperature."


# Plant Analytics Service

In [14]:
# ------------------- Plant analytics and health status functions ---------------
# Responsible for all analytics and health calculations derived from sensor data.
# Sensor Service provides raw DataFrames; Plant Analytics Service processes them.
# Dashboard UI calls these functions and renders the returned results.
# All timestamp comparisons use UTC to match the project-wide convention.

"""
Function to get the latest valid numeric sensor reading from a feed DataFrame.
input: data (DataFrame containing sensor readings with a "value" column), value_column (string name of the column containing the sensor values, default "value")
output: the latest valid numeric sensor reading (float) or None if no valid reading is available
"""
def get_latest_sensor_value(data, value_column="value"):
    if data is None or not isinstance(data, pd.DataFrame) or data.empty:
        return None
    if value_column not in data.columns:
        return None
    if "created_at" in data.columns:
        sorted_df = data.dropna(subset=[value_column]).sort_values(
            "created_at", ascending=False
        )
    else:
        sorted_df = data.dropna(subset=[value_column])
    if sorted_df.empty:
        return None
    try:
        return float(sorted_df.iloc[0][value_column])
    except (TypeError, ValueError):
        return None


"""
Function to filter a DataFrame of sensor readings to include only those from the current UTC calendar day.
Timestamps are parsed with utc=True so naive timestamps are treated as UTC, which is consistent with the project's use of datetime.timezone.utc elsewhere.
input: data (DataFrame containing sensor readings with a timestamp column), timestamp_column (string name of the column containing timestamps, default "created_at")
output: DataFrame filtered to include only rows from the current UTC calendar day, or an empty DataFrame if input is invalid.
"""
def filter_readings_for_today(data, timestamp_column="created_at"):
    if data is None or not isinstance(data, pd.DataFrame) or data.empty:
        return pd.DataFrame()
    if timestamp_column not in data.columns:
        return pd.DataFrame()
    today_utc = datetime.datetime.now(datetime.timezone.utc).date()
    try:
        ts = pd.to_datetime(data[timestamp_column], utc=True, errors="coerce")
        mask = ts.dt.date == today_utc
        return data[mask.fillna(False)].copy()
    except Exception:
        return pd.DataFrame()

"""
Function to calculate the average temperature and count of valid readings for the current UTC calendar day from a DataFrame of sensor readings.
input: data (DataFrame containing sensor readings with a "value" column and a timestamp column)
output: tuple (average temperature as a float rounded to 1 decimal place, count of valid readings as an int) for today's valid temperature readings;
returns (None, 0) if no valid data is available.
"""
def calculate_daily_average_temperature(data):
    today_df = filter_readings_for_today(data)
    if today_df.empty or "value" not in today_df.columns:
        return None, 0
    values = pd.to_numeric(today_df["value"], errors="coerce").dropna()
    if values.empty:
        return None, 0
    return round(float(values.mean()), 1), int(len(values))

"""
Function to calculate temperature fluctuation, trend, and count of valid readings over the last N hours from a DataFrame of sensor readings.
input: data (DataFrame containing sensor readings with a "value" column and a timestamp column), hours (int number of past hours to include in the calculation, default 6)
output: dict with keys: min (float), max (float), fluctuation (float), trend (string "rising"/"falling"/"stable"), count (int) describing temperature behavior over the last N hours;
returns a dict with None values and count 0 if fewer than 2 valid readings exist in the window or if input is invalid.
"""
def calculate_temperature_fluctuation(data, hours=6):
    empty = {"min": None, "max": None, "fluctuation": None, "trend": None, "count": 0}
    if data is None or not isinstance(data, pd.DataFrame) or data.empty:
        return empty
    if "value" not in data.columns or "created_at" not in data.columns:
        return empty
    try:
        cutoff = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(hours=hours)
        ts = pd.to_datetime(data["created_at"], utc=True, errors="coerce")
        window_df = data[ts.fillna(pd.Timestamp.min.tz_localize("UTC")) >= cutoff].copy()
        if window_df.empty:
            return empty
        values = pd.to_numeric(window_df["value"], errors="coerce").dropna()
        count = int(len(values))
        if count < 2:
            return {**empty, "count": count}

        min_temp    = round(float(values.min()), 1)
        max_temp    = round(float(values.max()), 1)
        fluctuation = round(max_temp - min_temp, 1)

        window_df["_ts"] = pd.to_datetime(window_df["created_at"], utc=True, errors="coerce")
        ordered   = window_df.dropna(subset=["_ts"]).sort_values("_ts")
        first_val = pd.to_numeric(ordered.iloc[0]["value"],  errors="coerce")
        last_val  = pd.to_numeric(ordered.iloc[-1]["value"], errors="coerce")

        if pd.isna(first_val) or pd.isna(last_val):
            trend = "stable"
        elif (last_val - first_val) > TEMPERATURE_STABLE_DELTA:
            trend = "rising"
        elif (first_val - last_val) > TEMPERATURE_STABLE_DELTA:
            trend = "falling"
        else:
            trend = "stable"

        return {
            "min": min_temp, "max": max_temp,
            "fluctuation": fluctuation, "trend": trend, "count": count,
        }
    except Exception:
        return empty

"""
Function to calculate the plant health status based on the latest temperature and humidity readings, using predefined thresholds.
input: temperature (float or None), humidity (float or None)
output: dict with keys: level (string "green"/"yellow"/"orange"/"red"/"unknown"), label (string), message (string), data_status (string "complete"/"partial"/"missing") describing the plant health status;
"""
def calculate_plant_health_status(temperature, humidity):
    t_cfg = HEALTH_THRESHOLDS["temperature"]
    h_cfg = HEALTH_THRESHOLDS["humidity"]

    def _classify(val, cfg):
        if val is None:
            return None
        try:
            val = float(val)
        except (TypeError, ValueError):
            return None
        g_min, g_max = cfg["green_min"], cfg["green_max"]
        if g_min <= val <= g_max:
            return "green"
        deviation = max(g_min - val, val - g_max, 0)
        if deviation <= cfg["yellow_margin"]:
            return "yellow"
        if deviation <= cfg["orange_margin"]:
            return "orange"
        return "red"

    t_level = _classify(temperature, t_cfg)
    h_level = _classify(humidity,    h_cfg)

    LEVEL_RANK = {"green": 0, "yellow": 1, "orange": 2, "red": 3}
    LABELS     = {"green": "Healthy", "yellow": "Needs Attention",
                  "orange": "Warning", "red": "Critical"}
    METRIC_MSG = {
        "green":   "{m} is within the configured range",
        "yellow":  "{m} is slightly outside the configured range",
        "orange":  "{m} is moderately outside the configured range",
        "red":     "{m} is far outside the configured range",
    }

    t_valid = t_level is not None
    h_valid = h_level is not None

    if not t_valid and not h_valid:
        return {
            "level":       "unknown",
            "label":       "Unknown",
            "message":     "Temperature and humidity data are currently unavailable.",
            "data_status": "missing",
        }

    if t_valid and h_valid:
        worst = max((t_level, h_level), key=lambda x: LEVEL_RANK[x])
        parts = [
            METRIC_MSG[t_level].format(m="Temperature"),
            METRIC_MSG[h_level].format(m="Humidity"),
        ]
        message = ". ".join(p[0].upper() + p[1:] for p in parts) + "."
        data_status = "complete"
    elif t_valid:
        worst = t_level
        base = METRIC_MSG[t_level].format(m="Temperature")
        message = (base[0].upper() + base[1:] +
                   ". Health status is based on temperature only "
                   "because humidity data is unavailable.")
        data_status = "partial"
    else:
        worst = h_level
        base = METRIC_MSG[h_level].format(m="Humidity")
        message = (base[0].upper() + base[1:] +
                   ". Health status is based on humidity only "
                   "because temperature data is unavailable.")
        data_status = "partial"

    return {
        "level":       worst,
        "label":       LABELS[worst],
        "message":     message,
        "data_status": data_status,
    }


# Journal Service

In [15]:
#------------------- Journal event types and validation functions ---------------

JOURNAL_EVENT_TYPES = {
    "watering":         "Watering",
    "fertilizing":      "Fertilizing",
    "leaf_inspection":  "Leaf Inspection",
    "problem_detected": "Problem Detected",
    "pest_observed":    "Pest Observed",
    "general_note":     "General Note",
}
JOURNAL_EVENTS_FULL = [
    {"type": "watering",         "label": "Watering",         "icon": "💧", "desc": "Routine or targeted watering"},
    {"type": "fertilizing",      "label": "Fertilizing",      "icon": "🌿", "desc": "Applied fertilizer or nutrients"},
    {"type": "leaf_inspection",  "label": "Leaf Inspection",  "icon": "🔍", "desc": "Visual check of leaves and stems"},
    {"type": "problem_detected", "label": "Problem Detected", "icon": "⚠️", "desc": "Spotted disease, stress, or damage"},
    {"type": "pest_observed",    "label": "Pest Observed",    "icon": "🐛", "desc": "Insects or pests noticed on plant"},
    {"type": "general_note",     "label": "General Note",     "icon": "📝", "desc": "Any other observation or reminder"},
]
JOURNAL_EVENTS_DICT = {e["type"]: e for e in JOURNAL_EVENTS_FULL}
JOURNAL_EVENT_ICONS = {e["type"]: e["icon"] for e in JOURNAL_EVENTS_FULL}
JOURNAL_NOTE_MAX_LEN = 500

"""
Function to validate the event type and note for a journal entry before saving it to Firebase.
input: event_type (string identifier for the type of journal event), note (string containing the user's note or description of the event)
output: tuple (is_valid: bool, error_message: string) where is_valid is True if the event_type is recognized and the note length is within limits, otherwise False; error_message contains a description of the validation failure if is_valid is False, or an empty string if validation passes.
"""
def validate_journal_event(event_type, note):
    if event_type not in JOURNAL_EVENT_TYPES:
        return False, f"Unknown event type: {event_type!r}"
    if len(note) > JOURNAL_NOTE_MAX_LEN:
        return False, f"Note is too long ({len(note)} chars; max {JOURNAL_NOTE_MAX_LEN})."
    return True, ""

"""
Function to convert an ISO-8601 UTC timestamp string to a human-readable display format for journal entries.
input: iso_str (string containing an ISO-8601 formatted UTC timestamp, e.g., "2026-06-08T18:30:00Z")
output: string formatted as "Month Day, Year at HH:MM" (e.g., "June 8, 2026 at 18:30") if parsing is successful; otherwise returns the original input string unchanged.
"""
def format_journal_datetime(iso_str):
    if not iso_str:
        return str(iso_str)
    try:
        dt = datetime.datetime.fromisoformat(str(iso_str).replace("Z", "+00:00"))
        return dt.strftime("%B %-d, %Y at %H:%M")
    except Exception:
        try:
            import re as _re
            clean = _re.sub(r"[+-]\d{2}:\d{2}$", "", str(iso_str)).rstrip("Z").strip()
            dt2 = datetime.datetime.strptime(clean[:16], "%Y-%m-%dT%H:%M")
            return dt2.strftime("%B %-d, %Y at %H:%M")
        except Exception:
            return str(iso_str)

"""
Function to validate, build, and persist a new journal event to Firebase.
It first validates the event type and note, then constructs a payload with the event details and current timestamp, and finally calls the Firebase function to save the event.
input: event_type (string identifier for the type of journal event), note (string containing the user's note or description of the event), source (string indicating the source of the event, default "manual")
output: dict with keys {ok: bool, key: str} if the event is successfully saved to Firebase, or {ok: bool, error: str, validation_error: bool} if validation fails or if there is an error saving to Firebase.
 The validation_error key is True if the failure was due to validation, and False if it was due to a Firebase error.
"""
def save_journal_event(event_type, note, source="manual"):
    ok, err_msg = validate_journal_event(event_type, note)
    if not ok:
        return {"ok": False, "error": err_msg, "validation_error": True}

    event_label = JOURNAL_EVENT_TYPES.get(event_type, event_type)
    payload = {
        "event_type":  event_type,
        "event_label": event_label,
        "note":        note.strip(),
        "created_at":  datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "source":      source,
    }
    return firebase_add_journal_event(payload)

"""
Function to delete a journal event by delegating the deletion operation to the Firebase Service.
The function receives the unique Firebase key stored in the event's "_key" field.
input: event_key (string containing the unique Firebase key of the journal event)
output: dict with keys {ok: bool} if the deletion succeeds,
or {ok: bool, error: str} if the key is missing or the Firebase deletion fails
"""
def delete_journal_event(event_key):
    if not event_key:
        return {
            "ok": False,
            "error": "Event key is missing."
        }

    return firebase_delete_journal_event(event_key)

"""
Function to retrieve, format, and sort journal events from Firebase for display on the dashboard.
input: limit (int number of recent events to retrieve, default 5)
output: list of dicts representing journal events, sorted by creation time in descending order, limited to the specified number of records;
returns an empty list if no events are available or if there is an error retrieving from Firebase.
"""
def get_formatted_journal_events(limit=5):
    raw = firebase_get_journal_events()
    if not raw:
        return []
    raw.sort(key=lambda e: str(e.get("created_at") or ""), reverse=True)
    return raw[:limit]


# Shared UI Styles and Components

In [16]:
APP_CSS = """
<style>
/* ===== WombatCombat Plant Care — modern dashboard theme v2 ===== */
.wb-app, .wb-app * { box-sizing: border-box; }

:root {
    --wb-green-900: #0f3d2a;
    --wb-green-700: #1c7a51;
    --wb-green-500: #28a96d;
    --wb-green-300: #74d6a3;
    --wb-mint:      #eafaf0;
    --wb-accent:    #f2a93b;
    --wb-ink:       #122019;
    --wb-muted:     #6b7c72;
    --wb-line:      #e3efe8;
    --wb-card:      #ffffff;
    --wb-radius:    22px;
    --wb-shadow:    0 2px 6px rgba(15,61,42,.05), 0 18px 40px rgba(15,61,42,.09);
    --wb-shadow-sm: 0 1px 3px rgba(15,61,42,.06), 0 8px 20px rgba(15,61,42,.06);
    --wb-font: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
}

.wb-app {
    font-family: var(--wb-font);
    direction: ltr;
    color: var(--wb-ink);
    background:
        radial-gradient(900px 360px at 110% -10%, rgba(116,214,163,.30), transparent 60%),
        radial-gradient(700px 320px at -10% 0%, rgba(242,169,59,.14), transparent 55%),
        linear-gradient(160deg, #ecfaf1 0%, #fffdf7 100%);
    border-radius: 30px;
    padding: clamp(14px, 2.4vw, 26px);
    border: 1px solid var(--wb-line);
}

/* ===== Header ===== */
.wb-header {
    position: relative;
    overflow: hidden;
    background: linear-gradient(130deg, var(--wb-green-900) 0%, var(--wb-green-700) 52%, var(--wb-green-500) 100%);
    color: #fff;
    border-radius: 26px;
    padding: clamp(22px, 3vw, 34px);
    margin-bottom: 20px;
    box-shadow: 0 22px 46px rgba(15,61,42,.30);
}
.wb-header::before {
    content: "";
    position: absolute; inset: 0;
    background:
        radial-gradient(360px 200px at 92% -30%, rgba(255,255,255,.30), transparent 70%),
        radial-gradient(280px 200px at 8% 130%, rgba(116,214,163,.45), transparent 70%);
    pointer-events: none;
}
.wb-head-row { position: relative; display: flex; align-items: center; gap: 16px; flex-wrap: wrap; }
.wb-logo {
    width: 58px; height: 58px; flex: 0 0 auto;
    border-radius: 18px;
    background: rgba(255,255,255,.16);
    backdrop-filter: blur(6px);
    border: 1px solid rgba(255,255,255,.35);
    display: flex; align-items: center; justify-content: center;
    font-size: 30px;
    box-shadow: 0 8px 20px rgba(0,0,0,.18);
}
.wb-head-text h1 {
    margin: 0;
    font-size: clamp(22px, 3.4vw, 34px);
    font-weight: 800;
    letter-spacing: -.025em;
    line-height: 1.05;
}
.wb-head-text p {
    margin: 8px 0 0;
    max-width: 62ch;
    opacity: .92;
    line-height: 1.5;
    font-size: clamp(13px, 1.4vw, 15px);
}
.wb-pills { position: relative; display: flex; flex-wrap: wrap; gap: 8px; margin-top: 16px; }
.wb-pill {
    display: inline-flex; align-items: center; gap: 6px;
    padding: 6px 13px;
    border-radius: 999px;
    background: rgba(255,255,255,.14);
    border: 1px solid rgba(255,255,255,.30);
    color: #fff;
    font-size: 12.5px; font-weight: 600;
    backdrop-filter: blur(4px);
}

/* ===== Cards ===== */
.card {
    position: relative;
    background: var(--wb-card);
    border: 1px solid var(--wb-line);
    border-radius: var(--wb-radius);
    padding: clamp(18px, 2vw, 24px);
    margin: 16px 0;
    box-shadow: var(--wb-shadow);
    overflow: hidden;
}
.card::before {
    content: "";
    position: absolute; top: 0; left: 0; right: 0; height: 4px;
    background: linear-gradient(90deg, var(--wb-green-500), var(--wb-green-300), var(--wb-accent));
}
.section-title {
    display: flex; align-items: center; gap: 10px;
    color: var(--wb-green-900);
    margin: 4px 0 12px;
    font-size: clamp(18px, 2.2vw, 24px);
    font-weight: 800;
    letter-spacing: -.015em;
}
.muted { color: var(--wb-muted); font-size: 14px; line-height: 1.55; }
.card p, .result-card p, .rag-box p { line-height: 1.55; overflow-wrap: anywhere; }
.card b, .result-card b, .rag-box b { color: var(--wb-green-900); }

/* ===== Metric grid (responsive auto-fit) ===== */
.metric-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
    gap: 14px;
    margin: 16px 0 6px;
}
.metric {
    position: relative;
    background: linear-gradient(180deg, #ffffff, #f4fcf7);
    border: 1px solid var(--wb-line);
    border-radius: 18px;
    padding: 16px 16px 18px;
    overflow: hidden;
    transition: transform .2s ease, box-shadow .2s ease;
}
.metric::before {
    content: ""; position: absolute; left: 0; top: 0; bottom: 0; width: 4px;
    background: linear-gradient(180deg, var(--wb-green-500), var(--wb-green-300));
}
.metric:hover { transform: translateY(-4px); box-shadow: var(--wb-shadow-sm); }
.metric-top { display: flex; align-items: center; gap: 9px; }
.metric-icon {
    width: 30px; height: 30px; flex: 0 0 auto;
    display: flex; align-items: center; justify-content: center;
    border-radius: 10px;
    background: var(--wb-mint);
    font-size: 16px;
}
.metric .label {
    color: var(--wb-muted);
    font-size: 11.5px; font-weight: 700;
    text-transform: uppercase; letter-spacing: .05em;
}
.metric .value {
    color: var(--wb-green-900);
    font-weight: 800;
    font-size: clamp(24px, 3vw, 30px);
    margin-top: 10px; line-height: 1;
}

/* ===== Badges ===== */
.badge {
    display: inline-flex; align-items: center; gap: 6px;
    padding: 6px 13px;
    border-radius: 999px;
    font-weight: 700; font-size: 12.5px; line-height: 1;
}
.badge::before { content: ""; width: 7px; height: 7px; border-radius: 50%; background: currentColor; }
.green  { background: #e4f8ed; color: #137a48; border: 1px solid #b8e6c9; }
.orange { background: #fff2db; color: #95560a; border: 1px solid #f1cf94; }
.red    { background: #ffe7e7; color: #b22222; border: 1px solid #f3b2b2; }

/* ===== Result + RAG ===== */
.result-card {
    background: #fcfffd;
    border: 1px solid var(--wb-line);
    border-left: 4px solid var(--wb-green-500);
    border-radius: 16px;
    padding: 16px 18px;
    margin: 12px 0;
    box-shadow: var(--wb-shadow-sm);
    transition: transform .18s ease, box-shadow .18s ease;
}
.result-card:hover { transform: translateY(-2px); box-shadow: var(--wb-shadow); border-left-color: var(--wb-accent); }
.result-card h3 { margin: 0 0 8px; color: var(--wb-green-900); font-size: 16px; }
.rag-box {
    position: relative;
    background: linear-gradient(180deg, #effaf3, #f7fdfa);
    border: 1px solid #c7ead4;
    border-radius: 20px;
    padding: 20px 22px;
    margin-top: 18px;
    box-shadow: var(--wb-shadow-sm);
}
.rag-box::before {
    content: ""; position: absolute; top: 0; left: 0; right: 0; height: 4px;
    background: linear-gradient(90deg, var(--wb-accent), var(--wb-green-500));
    border-radius: 20px 20px 0 0;
}
.rag-box h3 { margin: 0 0 10px; color: var(--wb-green-900); }
.task {
    display: flex; align-items: center; gap: 10px;
    padding: 12px 14px;
    background: #fbfffc;
    border: 1px solid var(--wb-line);
    border-radius: 14px;
    margin: 8px 0;
}

/* ===== ipywidgets buttons ===== */
.widget-button {
    border-radius: 14px !important;
    border: 1px solid var(--wb-line) !important;
    font-weight: 700 !important;
    box-shadow: var(--wb-shadow-sm) !important;
    transition: transform .15s ease, box-shadow .2s ease, filter .2s ease !important;
}
.widget-button:hover { transform: translateY(-2px); filter: brightness(1.04); box-shadow: var(--wb-shadow) !important; }
.widget-button:active { transform: translateY(0); }

/* ===== Inputs ===== */
.widget-text input, .widget-text textarea {
    border-radius: 14px !important;
    border: 1px solid var(--wb-line) !important;
    padding: 11px 15px !important;
    background: #ffffff !important;
    color: #15241c !important;
    transition: border-color .15s ease, box-shadow .15s ease !important;
}
.widget-text input::placeholder, .widget-text textarea::placeholder {
    color: #9aa8a0 !important;
    opacity: 1 !important;
}

/* ===== Chatbot ===== */
.chat-page {
    background: #fff;
    border: 1px solid var(--wb-line);
    border-radius: 24px;
    overflow: hidden;
    box-shadow: var(--wb-shadow);
}
.chat-header {
    background: linear-gradient(130deg, var(--wb-green-900), var(--wb-green-500));
    color: #fff;
    padding: 18px 22px;
    display: flex; align-items: center; gap: 14px;
}
.chat-avatar {
    width: 46px; height: 46px; flex: 0 0 auto;
    border-radius: 50%;
    background: rgba(255,255,255,.92);
    color: var(--wb-green-700);
    display: flex; align-items: center; justify-content: center;
    font-size: 23px;
    box-shadow: 0 6px 16px rgba(0,0,0,.16);
}
.chat-title { font-size: 18px; font-weight: 800; }
.chat-subtitle { font-size: 12.5px; opacity: .92; margin-top: 3px; }
.chat-messages {
    height: clamp(250px, 44vh, 380px);
    overflow-y: auto;
    padding: 20px;
    background: linear-gradient(180deg, #f3f8f4, #edf5ef);
    scroll-behavior: smooth;
}
.message-row { display: flex; margin-bottom: 13px; }
.message-row.user { justify-content: flex-end; }
.message-row.bot  { justify-content: flex-start; }
.message-bubble {
    max-width: 78%;
    padding: 12px 16px;
    border-radius: 18px;
    font-size: 14px; line-height: 1.5;
    box-shadow: 0 2px 10px rgba(0,0,0,.07);
    overflow-wrap: anywhere;
    animation: wb-pop .2s ease;
}
@keyframes wb-pop { from { opacity: 0; transform: translateY(6px); } to { opacity: 1; transform: none; } }
.message-row.user .message-bubble {
    background: linear-gradient(135deg, var(--wb-green-700), var(--wb-green-500));
    color: #fff; border-bottom-right-radius: 5px;
}
.message-row.bot .message-bubble {
    background: #fff; color: #1f2937;
    border: 1px solid var(--wb-line); border-bottom-left-radius: 5px;
}
.chat-controls { margin-top: 12px; }
.chat-messages::-webkit-scrollbar { width: 9px; }
.chat-messages::-webkit-scrollbar-thumb { background: #c6dbcd; border-radius: 999px; }

/* ===== Small screens ===== */
@media (max-width: 640px) {
    .message-bubble { max-width: 90%; }
    .metric-grid { gap: 10px; }
}

/* ===== Connection status bar ===== */
.wb-status {
    position: absolute;
    bottom: 18px;
    right: 22px;
    display: flex;
    flex-direction: row;
    align-items: center;
    gap: 10px;
    z-index: 2;
}
.wb-stat {
    display: inline-flex; align-items: center; gap: 8px;
    padding: 8px 14px;
    border-radius: 999px;
    background: rgba(255,255,255,.14);
    border: 1px solid rgba(255,255,255,.28);
    color: #fff;
    font-size: 12.5px; font-weight: 600;
    backdrop-filter: blur(4px);
    white-space: nowrap;
}
.wb-stat .dot {
    width: 9px; height: 9px; border-radius: 50%;
    flex: 0 0 auto;
    box-shadow: 0 0 0 0 currentColor;
}
.wb-stat.on  .dot { background: #5ef0a0; color: #5ef0a0; animation: wb-pulse 1.8s ease-out infinite; }
.wb-stat.off .dot { background: #ffb4b4; color: #ffb4b4; }
@keyframes wb-pulse {
    0%   { box-shadow: 0 0 0 0 rgba(94,240,160,.55); }
    100% { box-shadow: 0 0 0 7px rgba(94,240,160,0); }
}

@media (max-width: 760px) {
    .wb-status {
        position: static;
        flex-direction: row;
        flex-wrap: wrap;
        align-items: flex-start;
        margin-top: 14px;
    }
}

/* ===== Forced light canvas — immune to Colab dark mode ===== */
.wb-canvas {
    background: linear-gradient(160deg, #ecfaf1 0%, #fffdf7 100%) !important;
    border-radius: 24px !important;
    padding: 16px !important;
    margin-top: 10px !important;
    border: 1px solid var(--wb-line) !important;
}
.wb-canvas, .wb-canvas * { color: var(--wb-ink); }
.wb-canvas .widget-button { color: #15241c !important; }
.wb-canvas .widget-label { color: var(--wb-green-900) !important; }

body, .output_area, .jp-OutputArea, .jp-Cell-outputWrapper,
div.output_subarea, .colab-df-container {
    background: linear-gradient(160deg, #ecfaf1 0%, #fffdf7 100%) !important;
}
.wb-app { margin-bottom: 14px !important; }

/* ===== Search layout: results + sticky Gemini side panel ===== */
.search-layout {
    display: flex;
    flex-direction: row;
    align-items: flex-start;
    gap: 16px;
}
.search-results-col { flex: 1 1 0; min-width: 0; }
.search-aside {
    flex: 0 0 440px;
    position: sticky;
    top: 14px;
    align-self: flex-start;
}
.search-aside .rag-box { margin-top: 0; }
.search-aside .rag-box .ai-scroll {
    max-height: 70vh;
    overflow-y: auto;
    padding-right: 4px;
}
.search-aside .rag-box .ai-scroll::-webkit-scrollbar { width: 8px; }
.search-aside .rag-box .ai-scroll::-webkit-scrollbar-thumb { background: #c6dbcd; border-radius: 999px; }
.ai-placeholder {
    color: var(--wb-muted);
    font-size: 13.5px;
    line-height: 1.5;
}

/* ===== Plant Journal ===== */
.journal-event-card {
    position: relative;
    background: var(--wb-card);
    border: 1px solid var(--wb-line);
    border-radius: 18px;
    padding: 14px 18px;
    margin: 10px 0;
    box-shadow: var(--wb-shadow-sm);
    display: flex;
    gap: 14px;
    align-items: flex-start;
    transition: transform .18s ease, box-shadow .18s ease;
}
.journal-event-card:hover { transform: translateY(-2px); box-shadow: var(--wb-shadow); }
.journal-event-icon {
    font-size: 22px;
    flex: 0 0 auto;
    width: 40px; height: 40px;
    display: flex; align-items: center; justify-content: center;
    background: var(--wb-mint);
    border-radius: 12px;
}
.journal-event-body  { flex: 1 1 auto; min-width: 0; }
.journal-event-label { font-weight: 800; color: var(--wb-green-900); font-size: 15px; }
.journal-event-time  { color: var(--wb-muted); font-size: 12px; margin-bottom: 3px; }
.journal-event-note  { font-size: 14px; color: var(--wb-ink); margin-top: 4px; line-height: 1.5; }
.journal-empty {
    text-align: center; color: var(--wb-muted);
    padding: 28px 16px; font-size: 14px;
    border: 2px dashed var(--wb-line);
    border-radius: 16px; margin: 12px 0;
}

/* Journal event card-select buttons */
.widget-button.jcard {
    border-radius: 18px !important;
    min-height: 58px !important;
    border: 2px solid #dce8df !important;
    font-size: 13.5px !important;
    font-weight: 700 !important;
    white-space: nowrap !important;
    padding: 0 16px !important;
    background: #ffffff !important;
    color: #122019 !important;
    box-shadow: 0 1px 4px rgba(15,61,42,.05) !important;
    transition: all .15s ease !important;
    text-align: left !important;
}
.widget-button.jcard:hover {
    border-color: #74d6a3 !important;
    background: #f2fbf6 !important;
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 16px rgba(15,61,42,.11) !important;
}
.widget-button.jcard.jcard-sel {
    border-color: #28a96d !important;
    background: #e4f8ed !important;
    color: #0f3d2a !important;
    box-shadow: 0 0 0 3px rgba(40,169,109,.16), 0 4px 12px rgba(15,61,42,.08) !important;
}

/* Fix textarea background — Colab dark-mode override */
.widget-textarea textarea {
    background: #ffffff !important;
    color: #122019 !important;
    border: 1.5px solid #c8e6d4 !important;
    border-radius: 14px !important;
    padding: 12px 14px !important;
    font-size: 14px !important;
    line-height: 1.55 !important;
}
.widget-textarea textarea::placeholder { color: #9aa8a0 !important; opacity: 1 !important; }

/* ===== Dashboard additions ===== */

/* Plant info / metric info card — used in top and bottom rows */
.wb-info-row {
    display: flex;
    gap: 14px;
    flex-wrap: wrap;
    margin: 0 0 16px;
}
.wb-info-card {
    background: #fafdfb;
    border: 1px solid var(--wb-line);
    border-radius: var(--wb-radius);
    padding: 18px 20px;
    flex: 1 1 160px;
    min-width: 148px;
    box-shadow: var(--wb-shadow-sm);
    position: relative;
    overflow: hidden;
}
.wb-info-card::before {
    content: "";
    position: absolute; top: 0; left: 0; right: 0; height: 3px;
    background: linear-gradient(90deg, var(--wb-green-500), var(--wb-green-300));
}
.wb-info-card .ic-label {
    font-size: 11.5px;
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: .06em;
    color: var(--wb-muted);
    margin-bottom: 7px;
}
.wb-info-card .ic-value {
    font-size: 20px;
    font-weight: 800;
    color: var(--wb-green-900);
    line-height: 1.2;
}
.wb-info-card .ic-sub {
    font-size: 12px;
    color: var(--wb-muted);
    margin-top: 5px;
    line-height: 1.4;
}

/* Plant health status card */
.health-card {
    display: flex;
    align-items: flex-start;
    gap: 11px;
    padding: 10px 14px;
    border-radius: 14px;
    margin-top: 6px;
    border: 1px solid;
}
.health-dot {
    width: 13px; height: 13px;
    border-radius: 50%;
    flex: 0 0 auto;
    margin-top: 3px;
}
.health-text .ht-label { font-weight: 800; font-size: 14.5px; }
.health-text .ht-msg     { font-size: 12px; margin-top: 2px; line-height: 1.4; opacity: .9; }
.health-text .ht-partial { font-size: 11px; margin-top: 5px; opacity: .75; font-style: italic; }

.health-green   { background: #edfcf3; border-color: #a8e6c2; color: #0d5e35; }
.health-green   .health-dot { background: #28a96d; }
.health-yellow  { background: #fffce6; border-color: #eed870; color: #6e5100; }
.health-yellow  .health-dot { background: #d4a800; }
.health-orange  { background: #fff5e6; border-color: #f5bb7a; color: #7a3900; }
.health-orange  .health-dot { background: #e07820; }
.health-red     { background: #fff2f2; border-color: #f0a0a0; color: #820000; }
.health-red     .health-dot { background: #d42020; }
.health-unknown { background: #f6f6f6; border-color: #d0d0d0; color: #555; }
.health-unknown .health-dot { background: #aaa; }

/* Fluctuation trend indicators — neutral styling.
   Direction alone does not imply plant health risk.
   Health warnings are controlled only by calculate_plant_health_status. */
.trend-rising, .trend-falling, .trend-stable {
    color: var(--wb-ink);
    font-weight: 700;
    font-size: 14px;
}

/* Journal delete button */
.widget-button.journal-delete-button {
    border-radius: 12px !important;
    border: 1px solid #f1b8b8 !important;
    background: #fff1f1 !important;
    color: #a52828 !important;
    font-size: 12px !important;
    font-weight: 700 !important;
    box-shadow: none !important;
    transition: all 0.2s ease !important;
}

/* Journal delete button hover state */
.widget-button.journal-delete-button:hover {
    background: #ffe2e2 !important;
    border-color: #dc7777 !important;
    transform: translateY(-1px) !important;
}

/* Journal delete button disabled state */
.widget-button.journal-delete-button:disabled {
    opacity: 0.65 !important;
    cursor: not-allowed !important;
    transform: none !important;
}

</style>
"""

display(HTML(APP_CSS))# display the CSS styles for the dashboard

"""
Function to generate an HTML badge element with appropriate styling based on the provided text and tone.
input: text (string or value to be displayed inside the badge), tone (string indicating the tone of the badge, expected values are "green", "orange", or "red" for different health statuses)
output: string containing HTML markup for a styled badge element, where the text is safely escaped to prevent HTML injection, and the class of the badge is determined by the tone parameter to apply corresponding styles.
"""
def badge(text, tone):
    return f'<span class="badge {tone}">{html.escape(str(text))}</span>'


"""
Function to create an HTML card element for displaying a plant metric, including an icon, label, value, and optional unit.
The icon is chosen based on keywords in the label (temperature or humidity), and the entire content is safely escaped to prevent HTML injection.
input: label (string or value representing the name of the metric, e.g., "Temperature"), value (string or numeric value representing the metric's value, e.g., "22.5"), unit (optional string representing the unit of measurement, e.g., "°C")
output: string containing HTML markup for a styled card element that includes an icon corresponding to the metric type (temperature or humidity), the label, and the value with its unit, all properly escaped for safe display in a web context.
"""
def metric_card(label, value, unit=""):
    key = str(label).lower()
    icon = "\U0001F321️" if "temp" in key else "\U0001F4A8" if "humid" in key else "\U0001F4C8"
    return f"""
    <div class="metric">
        <div class="metric-top">
            <span class="metric-icon">{icon}</span>
            <div class="label">{html.escape(str(label))}</div>
        </div>
        <div class="value">{html.escape(str(value))}{html.escape(str(unit))}</div>
    </div>
    """

"""
Funtion that format a sensor value for display on the dashboard.
It attempts to convert the input value to a float and format it with one decimal place.
If the input value is missing (None) or cannot be converted to a float (e.g., it's a non-numeric string), the function returns a placeholder string "—" to indicate that the value is unavailable or invalid.
input: val (the sensor value to be formatted, which can be of any type)
output: a string representing the formatted sensor value with one decimal place if the input is a valid number, or "—" if the input is missing or non-numeric.
"""
def format_sensor_value(val):
    """Format a sensor value for display. Returns '—' if missing or non-numeric."""
    try:
        return f"{float(val):.1f}"
    except (TypeError, ValueError):
        return "—"


# Dashboard UI

In [17]:
#main output widget for the dashboard, where all the HTML content will be rendered
main_output = Output()


def _format_location():
    """Return a formatted location string from PLANT_PROFILE."""
    loc_type = PLANT_PROFILE.get("location_type", "")
    loc_name = PLANT_PROFILE.get("location_name", "")
    if loc_type and loc_name:
        return f"{loc_type} · {loc_name}"
    return loc_type or loc_name or "—"


def _render_health_card(health):
    """Return the inner HTML for the health status indicator."""
    level       = health.get("level", "unknown")
    label       = html.escape(health.get("label", "Unknown"))
    message     = html.escape(health.get("message", ""))
    data_status = health.get("data_status", "complete")
    partial_note = (
        '<div class="ht-partial">&#9888; Partial sensor data</div>'
        if data_status == "partial" else ""
    )
    return f'''<div class="health-card health-{level}">
        <div class="health-dot"></div>
        <div class="health-text">
            <div class="ht-label">{label}</div>
            <div class="ht-msg">{message}</div>
            {partial_note}
        </div>
    </div>'''


def _render_daily_avg_card(avg, count):
    """Return a .wb-info-card HTML block for the daily average temperature."""
    if avg is None:
        value_html = '<div class="ic-value" style="font-size:16px; color:var(--wb-muted);">—</div>'
        sub = "No temperature readings available today"
    else:
        value_html = (
            f'<div class="ic-value">'
            f'{avg:.1f}'
            f'<span style="font-size:15px; font-weight:600;">°C</span>'
            f'</div>'
        )
        plural = "s" if count != 1 else ""
        sub = f"Based on {count} reading{plural} today (UTC)"
    return f'''<div class="wb-info-card" style="flex: 1 1 200px;">
        <div class="ic-label">\U0001f321️ Daily Average</div>
        {value_html}
        <div class="ic-sub">{html.escape(sub)}</div>
    </div>'''


def _render_fluctuation_card(fluct):
    """Return a .wb-info-card HTML block for the temperature fluctuation summary."""
    hours = TEMPERATURE_FLUCTUATION_HOURS
    count = fluct.get("count", 0)

    if count < 2:
        if count == 0:
            sub = "No data in the selected time window."
        else:
            sub = "Not enough recent data to calculate fluctuation."
        return f'''<div class="wb-info-card" style="flex: 2 1 280px;">
            <div class="ic-label">\U0001f321️ Last {hours} Hours</div>
            <div class="ic-value" style="font-size:16px; color:var(--wb-muted);">—</div>
            <div class="ic-sub">{html.escape(sub)}</div>
        </div>'''

    trend = fluct.get("trend", "stable")
    TREND_MAP = {
        "rising":  ("↑ Rising",  "trend-rising"),
        "falling": ("↓ Falling", "trend-falling"),
        "stable":  ("→ Stable",  "trend-stable"),
    }
    trend_txt, trend_cls = TREND_MAP.get(trend, ("—", ""))
    plural = "s" if count != 1 else ""

    return f'''<div class="wb-info-card" style="flex: 2 1 280px;">
        <div class="ic-label">\U0001f321️ Last {hours} Hours</div>
        <div style="display:flex; gap:18px; flex-wrap:wrap; align-items:flex-start; margin-top:8px;">
            <div>
                <div style="font-size:11px; color:var(--wb-muted); text-transform:uppercase; letter-spacing:.05em;">Min</div>
                <div style="font-size:22px; font-weight:800; color:var(--wb-green-900); line-height:1.1;">{fluct["min"]:.1f}°C</div>
            </div>
            <div>
                <div style="font-size:11px; color:var(--wb-muted); text-transform:uppercase; letter-spacing:.05em;">Max</div>
                <div style="font-size:22px; font-weight:800; color:var(--wb-green-900); line-height:1.1;">{fluct["max"]:.1f}°C</div>
            </div>
            <div>
                <div style="font-size:11px; color:var(--wb-muted); text-transform:uppercase; letter-spacing:.05em;">Fluctuation</div>
                <div style="font-size:22px; font-weight:800; color:var(--wb-green-900); line-height:1.1;">{fluct["fluctuation"]:.1f}°C</div>
            </div>
            <div>
                <div style="font-size:11px; color:var(--wb-muted); text-transform:uppercase; letter-spacing:.05em;">Trend</div>
                <div class="{trend_cls}" style="margin-top:3px;">{html.escape(trend_txt)}</div>
            </div>
        </div>
        <div class="ic-sub" style="margin-top:8px;">Based on {count} reading{plural} ({hours}h window, UTC)</div>
    </div>'''


def build_dashboard_screen():
    """Render the enhanced plant dashboard with plant info, health status, and sensor metrics."""
    # show loading state:
    with main_output:
        clear_output(wait=True)
        display(HTML(APP_CSS))
        display(HTML('''
        <div class="card">
            <h2 class="section-title">\U0001f4ca Plant Dashboard</h2>
            <p class="muted">⏳ Loading sensor data, please wait…</p>
        </div>
        '''))

    # fetch sensor data:
    # Fetch 100 temperature readings to support daily average + fluctuation calc.
    temp_df,  _temp_err  = fetch_iot_feed(feed="temperature", limit=100)
    humid_df, _humid_err = fetch_iot_feed(feed="humidity",    limit=10)

    # Firebase cached summary as fallback when IoT server is unavailable.
    firebase_reading = firebase_get_latest_sensor_reading()

    # derive values
    latest_temp  = get_latest_sensor_value(temp_df)
    latest_humid = get_latest_sensor_value(humid_df)

    # Fall back to Firebase cached values if the IoT server did not respond.
    if latest_temp is None and firebase_reading:
        try:
            v = firebase_reading.get("temperature")
            latest_temp = float(v) if v is not None else None
        except (TypeError, ValueError):
            pass
    if latest_humid is None and firebase_reading:
        try:
            v = firebase_reading.get("air_humidity")
            latest_humid = float(v) if v is not None else None
        except (TypeError, ValueError):
            pass

    daily_avg, daily_count = calculate_daily_average_temperature(temp_df)
    fluct  = calculate_temperature_fluctuation(temp_df, hours=TEMPERATURE_FLUCTUATION_HOURS)
    health = calculate_plant_health_status(latest_temp, latest_humid)

    # Last updated string
    updated_str = "unknown"
    if temp_df is not None and not temp_df.empty and "created_at" in temp_df.columns:
        latest_ts = pd.to_datetime(temp_df["created_at"], utc=True, errors="coerce").max()
        if pd.notna(latest_ts):
            try:
                d = latest_ts.to_pydatetime()
                updated_str = d.strftime(f"%B {d.day}, %Y at %H:%M UTC")
            except Exception:
                updated_str = str(latest_ts)
    elif firebase_reading:
        updated_str = str(firebase_reading.get("updated_at", "unknown"))

    data_source_lbl = (
        "IoT server"     if temp_df is not None
        else "Firebase cache" if firebase_reading
        else "unavailable"
    )

    #render final dashboard
    with main_output:
        clear_output(wait=True)
        display(HTML(APP_CSS))

        # Row 0: Page title
        display(HTML(
            '<h2 class="section-title" style="margin:12px 0 14px;">'
            '\U0001f4ca Plant Dashboard</h2>'
        ))

        # Row 1: Plant info + health status
        location_str = _format_location()
        health_inner = _render_health_card(health)
        display(HTML(f'''
        <div class="wb-info-row">
            <div class="wb-info-card">
                <div class="ic-label">\U0001f33f Plant Type</div>
                <div class="ic-value">{html.escape(PLANT_PROFILE.get("plant_type", "—"))}</div>
            </div>
            <div class="wb-info-card">
                <div class="ic-label">\U0001f4cd Location</div>
                <div class="ic-value" style="font-size:17px;">{html.escape(location_str)}</div>
            </div>
            <div class="wb-info-card" style="flex: 2 1 240px;">
                <div class="ic-label">\U0001f49a Plant Health</div>
                {health_inner}
            </div>
        </div>
        '''))

        # Row 2: Current sensor readings
        temp_display  = format_sensor_value(latest_temp)
        humid_display = format_sensor_value(latest_humid)
        display(HTML(f'''
        <div class="card">
            <h3 class="section-title" style="font-size:16px; margin-bottom:12px;">
                \U0001f4e1 Current Readings
            </h3>
            <div class="metric-grid">
                {metric_card("Temperature", temp_display, "°C")}
                {metric_card("Air Humidity", humid_display, "%")}
            </div>
            <p class="muted" style="margin-top:10px; font-size:12.5px;">
                <b>Last updated:</b> {html.escape(updated_str)}
                &nbsp;·&nbsp;
                <b>Source:</b> {html.escape(data_source_lbl)}
            </p>
        </div>
        '''))

        # Row 3: Daily average + Fluctuation
        daily_card = _render_daily_avg_card(daily_avg, daily_count)
        fluct_card = _render_fluctuation_card(fluct)
        display(HTML(f'''
        <div class="wb-info-row">
            {daily_card}
            {fluct_card}
        </div>
        '''))

        # Temperature history chart
        if temp_df is not None and not temp_df.empty:
            try:
                chart_df = temp_df.copy()
                chart_df["_ts"] = pd.to_datetime(
                    chart_df["created_at"], utc=True, errors="coerce"
                )
                chart_df = (
                    chart_df.dropna(subset=["_ts", "value"])
                    .sort_values("_ts")
                )
                if not chart_df.empty:
                    display(HTML('''
                    <div class="card" style="padding-bottom:8px;">
                        <h3 class="section-title" style="font-size:16px; margin-bottom:4px;">
                            \U0001f4c8 Temperature History
                        </h3>
                        <p class="muted" style="font-size:12.5px; margin-bottom:8px;">
                            Recent temperature readings (UTC timestamps).
                        </p>
                    </div>
                    '''))
                    fig, ax = plt.subplots(figsize=(10, 3.5))
                    ax.plot(
                        chart_df["_ts"], chart_df["value"],
                        color="#28a96d", linewidth=1.8, marker="o", markersize=3,
                        label="Temperature (°C)"
                    )
                    ax.set_xlabel("Time (UTC)", fontsize=9)
                    ax.set_ylabel("Temperature (°C)", fontsize=9)
                    ax.set_title("Temperature over time", fontsize=11)
                    ax.grid(True, linestyle="--", alpha=0.45)
                    ax.legend(fontsize=9)
                    plt.xticks(rotation=30, fontsize=8)
                    plt.tight_layout()
                    plt.show()
            except Exception as _chart_err:
                display(HTML(
                    '<div class="card">'
                    '<p class="muted">Temperature chart could not be rendered.</p>'
                    '</div>'
                ))
                print(f"Dashboard chart error: {_chart_err}")
        elif temp_df is None:
            display(HTML('''
            <div class="card">
                <h3 class="section-title" style="font-size:16px;">\U0001f4c8 Temperature History</h3>
                <p class="muted">
                    Temperature data is unavailable. The sensor server could not be reached.
                    Current values are loaded from Firebase cache where available.
                </p>
            </div>
            '''))


# Journal UI

In [18]:

_journal_selected_event = "watering"

journal_card_buttons = {}

journal_delete_buttons = []

for event in JOURNAL_EVENTS_FULL:
    event_type = event["type"]
    event_icon = event["icon"]
    event_label = event["label"]

    button = widgets.Button(
        description=f"{event_icon}  {event_label}",
        layout=Layout(
            flex="1 1 calc(33% - 8px)",
            min_width="148px",
            height="58px",
            margin="4px 4px"
        ),
        style=ButtonStyle(),
    )

    button.add_class("jcard")
    journal_card_buttons[event_type] = button

journal_hint_html = widgets.HTML(value="")

journal_note_input = widgets.Textarea(
    value="",
    placeholder="Optional note (max 500 characters)...",
    description="",
    layout=Layout(width="100%", height="90px"),
)

journal_save_button = widgets.Button(
    description="💾 Save Event",
    style=ButtonStyle(button_color="#28a96d"),
    layout=Layout(width="160px", height="42px"),
)

journal_clear_button = widgets.Button(
    description="✕  Clear Form",
    style=ButtonStyle(button_color="#ffffff"),
    layout=Layout(width="140px", height="42px"),
)

journal_feedback_output = Output()
journal_list_output = Output()


def _set_journal_selection(event_type):
    global _journal_selected_event
    _journal_selected_event = event_type
    for k, btn in journal_card_buttons.items():
        if k == event_type:
            btn.style.button_color = "#e4f8ed"
            btn.add_class("jcard-sel")
        else:
            btn.style.button_color = "#ffffff"
            btn.remove_class("jcard-sel")
    info = JOURNAL_EVENTS_DICT.get(event_type, {})
    desc = info.get("desc", "")
    journal_hint_html.value = (
        f'<p style="font-size:13px; color:#6b7c72; margin:4px 0 0 2px;">{html.escape(desc)}</p>'
        if desc else ""
    )


def _make_journal_card_handler(event_type):
    def _handler(button):
        _set_journal_selection(event_type)
    return _handler


for _jkey in journal_card_buttons:
    journal_card_buttons[_jkey].on_click(_make_journal_card_handler(_jkey))

_set_journal_selection("watering")


def format_journal_datetime(iso_str):
    try:
        dt = datetime.datetime.fromisoformat(str(iso_str).replace("Z", "+00:00"))
        day = dt.day
        return dt.strftime(f"%B {day}, %Y at %H:%M")
    except Exception:
        return str(iso_str)


def _render_journal_card(event):
    event_type = event.get("event_type", "general_note")
    info       = JOURNAL_EVENTS_DICT.get(event_type, {})
    label      = html.escape(event.get("event_label", info.get("label", "Event")))
    icon       = info.get("icon", "📝")
    note       = (event.get("note") or "").strip()
    created    = event.get("created_at", "")
    time_str   = format_journal_datetime(created) if created else "Unknown time"
    note_html  = (
        f'<div class="journal-event-note">{html.escape(note)}</div>'
        if note else ""
    )
    return f'''
    <div class="journal-event-card">
        <div class="journal-event-icon">{icon}</div>
        <div class="journal-event-body">
            <div class="journal-event-time">{html.escape(time_str)}</div>
            <div class="journal-event-label">{label}</div>
            {note_html}
        </div>
    </div>'''

def make_delete_journal_handler(event_key):

    def handle_delete(button):
        button.disabled = True
        button.description = "Deleting..."

        journal_feedback_output.clear_output(wait=True)

        with journal_feedback_output:
            display(HTML(
                '<p class="muted">⏳ Deleting event...</p>'
            ))

        try:
            result = delete_journal_event(event_key)

            journal_feedback_output.clear_output(wait=True)

            with journal_feedback_output:
                if result.get("ok"):
                    display(HTML(
                        '<p style="color:#137a48;">'
                        '✅ Event deleted successfully.'
                        '</p>'
                    ))

                    refresh_journal_events()

                else:
                    error_message = html.escape(
                        result.get("error", "Unknown error")
                    )

                    display(HTML(
                        f'<p style="color:#b22222;">'
                        f'❌ Could not delete event: {error_message}'
                        f'</p>'
                    ))

                    button.disabled = False
                    button.description = "🗑 Delete"

        except Exception as exc:
            journal_feedback_output.clear_output(wait=True)

            with journal_feedback_output:
                display(HTML(
                    f'<p style="color:#b22222;">'
                    f'❌ Unexpected error: '
                    f'{html.escape(str(exc))}'
                    f'</p>'
                ))

            button.disabled = False
            button.description = "🗑 Delete"

    return handle_delete

def refresh_journal_events():
    global journal_delete_buttons
    journal_list_output.clear_output(wait=True)
    # Reset the button list before rebuilding the journal display.
    journal_delete_buttons = []
    with journal_list_output:
        display(HTML(APP_CSS))
        if not FIREBASE_READY:
            display(HTML('''
            <div class="card">
                <h3 class="section-title">📋 Recent Events</h3>
                <p class="muted">
                    Firebase is not connected.
                    Journal events cannot be loaded.
                </p>
            </div>
            '''))
            return
        events = get_formatted_journal_events(limit=5)
        if not events:
            display(HTML('''
            <div class="card">
                <h3 class="section-title">📋 Recent Events</h3>
                <div class="journal-empty">
                    No journal events yet.
                    Save your first event above.
                </div>
            </div>
            '''))
            return
        event_widgets = []
        for event in events:
            # The "_key" field was added when the events were loaded
            # from Firebase and identifies the exact record to delete.
            event_key = event.get("_key")

            delete_button = widgets.Button(
                description="🗑 Delete",
                tooltip="Delete this journal event",
                layout=Layout(
                    width="105px",
                    height="36px"
                ),
                style=ButtonStyle(
                    button_color="#fff1f1"
                )
            )
            delete_button.add_class("journal-delete-button")
            # Connect this button to a handler containing
            # the current event's Firebase key.
            delete_button.on_click(
                make_delete_journal_handler(event_key)
            )
            # Keep a reference to the button so it remains active
            # after the function finishes rendering the widgets.
            journal_delete_buttons.append(delete_button)

            event_content = widgets.HTML(
                value=_render_journal_card(event),
                layout=Layout(
                    flex="1 1 auto",
                    min_width="0"
                )
            )
            event_row = HBox(
                [
                    event_content,
                    delete_button
                ],
                layout=Layout(
                    width="100%",
                    align_items="center",
                    gap="10px"
                )
            )
            event_widgets.append(event_row)
        display(HTML('''
        <div class="card">
            <h3 class="section-title">📋 Recent Events</h3>
        </div>
        '''))

        display(VBox(
            event_widgets,
            layout=Layout(
                width="100%",
                gap="10px"
            )
        ))


def render_plant_journal():
    with main_output:
        clear_output(wait=True)
        display(HTML(APP_CSS))
        display(HTML('''
        <div class="card">
            <h2 class="section-title">📓 Plant Journal</h2>
            <p class="muted">
                Document actions and events for your Milk Thistle plant.
                Events are saved to Firebase and shown newest first.
            </p>
        </div>'''))
        _set_journal_selection(_journal_selected_event)
        display(HTML(
            '<div style="font-weight:700; color:#0f3d2a; font-size:12.5px; text-transform:uppercase;'
            ' letter-spacing:.06em; margin:18px 0 8px 2px;">➕ Log a New Event</div>'
        ))
        display(HTML(
            '<div style="font-weight:700; color:#0f3d2a; font-size:12.5px; text-transform:uppercase;'
            ' letter-spacing:.06em; margin:0 0 8px 2px;">Event Type</div>'
        ))
        display(VBox(
            [
                HBox(
                    [journal_card_buttons["watering"],
                     journal_card_buttons["fertilizing"],
                     journal_card_buttons["leaf_inspection"]],
                    layout=Layout(flex_flow="row wrap", gap="8px", width="100%"),
                ),
                HBox(
                    [journal_card_buttons["problem_detected"],
                     journal_card_buttons["pest_observed"],
                     journal_card_buttons["general_note"]],
                    layout=Layout(flex_flow="row wrap", gap="8px", width="100%"),
                ),
                journal_hint_html,
            ],
            layout=Layout(width="100%"),
        ))
        display(HTML(
            '<div style="font-weight:700; color:#0f3d2a; font-size:12.5px; text-transform:uppercase;'
            ' letter-spacing:.06em; margin:20px 0 6px 2px;">Notes</div>'
        ))
        display(journal_note_input)
        display(HBox(
            [journal_save_button, journal_clear_button],
            layout=Layout(margin="12px 0 0 0", gap="10px"),
        ))
        display(journal_feedback_output)
        refresh_journal_events()
        display(journal_list_output)


def handle_save_journal_event(button):
    """Save a journal event by delegating to the Journal Service."""
    event_type = _journal_selected_event
    note       = journal_note_input.value

    journal_save_button.disabled    = True
    journal_save_button.description = "Saving..."

    journal_feedback_output.clear_output(wait=True)
    with journal_feedback_output:
        display(HTML('<p class="muted">⏳ Saving event…</p>'))

    try:
        result = save_journal_event(event_type, note)

        journal_feedback_output.clear_output(wait=True)
        with journal_feedback_output:
            if result.get("ok"):
                display(HTML('<p style="color:#137a48;">✅ Event saved successfully.</p>'))
                journal_note_input.value = ""
                _set_journal_selection("watering")
                refresh_journal_events()
            else:
                err  = html.escape(result.get("error", "Unknown error"))
                icon = "⚠️" if result.get("validation_error") else "❌"
                display(HTML(f'<p style="color:#b22222;">{icon} {err}</p>'))
    except Exception as exc:
        journal_feedback_output.clear_output(wait=True)
        with journal_feedback_output:
            display(HTML(f'<p style="color:#b22222;">❌ Unexpected error: {html.escape(str(exc))}</p>'))
    finally:
        journal_save_button.disabled    = False
        journal_save_button.description = "💾 Save Event"

def handle_clear_journal_form(button):
    journal_note_input.value = ""
    _set_journal_selection("watering")
    journal_feedback_output.clear_output(wait=True)


def handle_refresh_journal(button):
    journal_feedback_output.clear_output(wait=True)
    with journal_feedback_output:
        display(HTML('<p class="muted">⏳ Loading journal...</p>'))
    refresh_journal_events()
    journal_feedback_output.clear_output(wait=True)


# Guard prevents duplicate registration when cell 31 is re-run without re-running cell 29.
if not getattr(journal_save_button, "_jcb_registered", False):
    journal_save_button.on_click(handle_save_journal_event)
    journal_clear_button.on_click(handle_clear_journal_form)
    journal_save_button._jcb_registered  = True
    journal_clear_button._jcb_registered = True


# Search UI

In [19]:
# ---------- Search / RAG screen ----------

search_input = widgets.Text(
    value="milk thistle mildew humidity",
    placeholder="Search: mildew, humidity, moisture, yellowing, disease...",
    description="Query:",
    style={"description_width": "60px"},
    layout=Layout(width="540px")
)

# Disabled while Gemini is generating an answer.
search_button = widgets.Button(
    description="Search",
    style=ButtonStyle(button_color="#2e7d5b"),
    layout=Layout(width="140px")
)


def build_search_results_html(results=None, answer=None, loading=False):
    """Return the results + RAG panel as a single HTML string."""
    if not results:
        results_html = """
        <div class="result-card">
            <h3>No matching articles found</h3>
            <p class="muted">Try terms such as mildew, disease, humidity, moisture, yellowing, leaf, or temperature.</p>
        </div>
        """
    else:
        cards = []
        for article, score, matched_terms in results:
            snippet = article["text"][:450].replace("\n", " ")
            cards.append(f"""
            <div class="result-card">
                <h3>{html.escape(article["title"])} <span class="muted">({article["id"]})</span></h3>
                <p><b>Authors:</b> {html.escape(article["authors"])}</p>
                <p><b>Score:</b> {score}</p>
                <p><b>Matched terms:</b> {html.escape(', '.join(matched_terms))}</p>
                <p>{html.escape(snippet)}...</p>
                <p class="muted">{html.escape(article["url"])}</p>
            </div>
            """)
        results_html = "".join(cards)

    if loading:
        ai_body = '<div class="ai-scroll"><p>⏳ Generating answer, please wait...</p></div>'
    elif answer is not None:
        safe_ans = html.escape(str(answer))
        safe_ans = safe_ans.replace("\n", "<br>")
        safe_ans = safe_ans.replace("&lt;b&gt;", "<b>").replace("&lt;/b&gt;", "</b>")
        safe_ans = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', safe_ans)
        ai_body = f'<div class="ai-scroll"><p>{safe_ans}</p></div>'
    else:
        ai_body = '<div class="ai-placeholder">The Gemini / RAG overview will appear here after a search.</div>'

    return f"""
    <div class="card">
        <h3>Search Results</h3>
        <p><b>Number of results:</b> {len(results or [])}</p>
    </div>
    <div class="search-layout">
        <div class="search-aside">
            <div class="rag-box">
                <h3>\U0001F9E0 Gemini / RAG Answer</h3>
                {ai_body}
            </div>
        </div>
        <div class="search-results-col">
            {results_html}
        </div>
    </div>
    """


def build_search_screen(results=None, query=None, answer=None, loading=False):
    """Draw the search screen into main_output. Uses single-output approach to avoid Colab nesting issues."""
    with main_output:
        clear_output(wait=True)
        display(HTML(APP_CSS))
        display(HTML("""
        <div class="card">
            <h2 class="section-title">\U0001F50D Academic Search Engine + RAG</h2>
            <p class="muted">
                Search the Milk Thistle academic article index.
                Gemini generates an answer from the retrieved context.
            </p>
        </div>
        """))
        display(HBox([search_input, search_button]))
        if query is not None:
            display(HTML(build_search_results_html(results=results, answer=answer, loading=loading)))


def handle_search(button):
    search_button.disabled = True
    search_button.description = "Searching..."
    try:
        query = search_input.value.strip()
        if not query:
            build_search_screen()
            with main_output:
                display(HTML('<div class="card">Please type a search query first.</div>'))
            return

        results, _ = search_articles(query, ARTICLES, FULL_INVERTED_INDEX)
        build_search_screen(results=results, query=query, loading=True)
        answer = rag_answer(query, results)
        build_search_screen(results=results, query=query, answer=answer)

    except Exception as e:
        print(f"Search error: {e}")
        build_search_screen()
        with main_output:
            display(HTML("""
            <div class="card">
                <h3>Search failed</h3>
                <p class="muted">An error occurred. Check the Colab output for details.</p>
            </div>
            """))
    finally:
        search_button.disabled = False
        search_button.description = "Search"


search_button.on_click(handle_search)


# Chatbot UI

In [20]:
# ---------- Chatbot screen ----------

chat_input = widgets.Text(
    value="",
    placeholder="Write a message...",
    description="",
    layout=Layout(width="100%")
)

chat_send_button = widgets.Button(
    description="Send",
    style=ButtonStyle(button_color="#2e7d5b"),
    layout=Layout(width="120px")
)

chat_clear_button = widgets.Button(
    description="Clear",
    style=ButtonStyle(button_color="#ffffff"),
    layout=Layout(width="100px")
)


def render_chat_messages():
    """Build the chat thread HTML string from CHAT_HISTORY."""
    parts = ["""
    <div class="chat-page">
        <div class="chat-header">
            <div class="chat-avatar">\U0001F33F</div>
            <div>
                <div class="chat-title">WombatCombat Milk Thistle Assistant</div>
                <div class="chat-subtitle">Ask about Milk Thistle care, sensors, disease and academic RAG</div>
            </div>
        </div>
        <div class="chat-messages">
    """]

    if not CHAT_HISTORY:
        parts.append("""
            <div class="message-row bot">
                <div class="message-bubble">
                    Hi! Ask me something like:
                    <br><br>
                    How does humidity affect Milk Thistle?
                    <br>
                    What should I check if the leaves are yellow?
                </div>
            </div>
        """)

    for message in CHAT_HISTORY:
        role = message["role"]
        safe_msg = html.escape(message["content"])
        safe_msg = safe_msg.replace("\n", "<br>")
        safe_msg = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', safe_msg)
        row_cls = "user" if role == "user" else "bot"
        parts.append(f"""
            <div class="message-row {row_cls}">
                <div class="message-bubble">{safe_msg}</div>
            </div>
        """)

    parts.append("</div></div>")
    return "".join(parts)


def build_chatbot_screen():
    with main_output:
        clear_output(wait=True)
        display(HTML(APP_CSS))
        display(HTML("""
        <div class="card">
            <h2 class="section-title">\U0001F4AC AI Chatbot</h2>
            <p class="muted">
                Uses Gemini when available, falls back to an NLTK rule-based chatbot.
                Also searches the academic articles and reads the latest sensor context.
            </p>
        </div>
        """))
        display(HTML(render_chat_messages()))
        display(HBox(
            [chat_input, chat_send_button, chat_clear_button],
            layout=Layout(width="100%", align_items="center", margin="12px 0 0 0")
        ))


def handle_chat_send(button):
    user_message = chat_input.value.strip()
    if not user_message:
        return
    chat_send_button.disabled = True
    chat_send_button.description = "Thinking..."
    try:
        CHAT_HISTORY.append({"role": "user", "content": user_message})
        try:
            answer = chatbot_answer(user_message)
            CHAT_HISTORY.append({"role": "bot", "content": answer})
        except Exception as e:
            print(f"Chatbot error: {e}")
            CHAT_HISTORY.append({"role": "bot", "content": "Something went wrong. Please try again."})
        chat_input.value = ""
        build_chatbot_screen()
    finally:
        chat_send_button.disabled = False
        chat_send_button.description = "Send"


def handle_chat_clear(button):
    CHAT_HISTORY.clear()
    build_chatbot_screen()





# ── Plant Journal handlers ───────────────────────────────────────────────────


chat_send_button.on_click(handle_chat_send)
chat_clear_button.on_click(handle_chat_clear)


# Upload Image UI

In [21]:
# ---------- Upload / Image Analysis screen ----------

upload_widget = widgets.FileUpload(accept="image/*", multiple=False, description="Choose image")
analyze_image_button = widgets.Button(
    description="Upload Image",
    style=ButtonStyle(button_color="#2e7d5b"),
    layout=Layout(width="170px")
)


def get_uploaded_file_name(file_upload_widget):
    """Return the uploaded file name, compatible with different ipywidgets versions."""
    value = file_upload_widget.value
    if isinstance(value, (tuple, list)) and len(value) > 0:
        return value[0].get("name", "uploaded image")
    if isinstance(value, dict) and len(value) > 0:
        return list(value.keys())[0]
    return None


def build_upload_screen(result_html=None):
    with main_output:
        clear_output(wait=True)
        display(HTML(APP_CSS))
        display(HTML("""
        <div class="card">
            <h2 class="section-title">\U0001F4F7 Plant Image Upload</h2>
            <p class="muted">
                Upload a photo of your plant.
                The Hugging Face image comparison feature is not connected yet.
                This screen is still incomplete.
            </p>
        </div>
        """))
        display(VBox([upload_widget, analyze_image_button]))
        if result_html:
            display(HTML(result_html))


def handle_analyze_image(button):
    file_name = get_uploaded_file_name(upload_widget)
    if not file_name:
        build_upload_screen(result_html='<div class="card">Please choose an image file first.</div>')
        return
    result_html = f"""
    <div class="result-card">
        <h3>Image uploaded: {html.escape(file_name)}</h3>
        <p>{badge("Image comparison: not connected yet", "orange")}</p>
        <p class="muted">
            The Hugging Face image comparison feature is not implemented yet.
            This screen is still incomplete.
        </p>
    </div>
    """
    build_upload_screen(result_html=result_html)


analyze_image_button.on_click(handle_analyze_image)


# Sensors UI

In [22]:
# ---------- Sensor / IoT screen ----------

iot_limit_dropdown = widgets.Dropdown(
    options=[("10 samples", 10), ("25 samples", 25), ("50 samples", 50), ("100 samples", 100)],
    value=10,
    description="Samples:",
    style={"description_width": "70px"},
    layout=Layout(width="220px")
)

iot_refresh_button = widgets.Button(
    description="\U0001F504 Refresh Sensor Data",
    style=ButtonStyle(button_color="#28a96d"),
    layout=Layout(width="220px", margin="0 0 0 12px")
)


def build_sensor_screen():
    with main_output:
        clear_output(wait=True)
        display(HTML(APP_CSS))
        display(HTML("""
        <div class="card">
            <div style="display:flex; justify-content:space-between; align-items:flex-start; gap:20px; flex-wrap:wrap;">
                <div>
                    <h2 class="section-title">\U0001F4E1 Plant Sensor Monitor</h2>
                    <p class="muted">
                        View the latest temperature and air-humidity
                        readings from the course IoT server.
                    </p>
                </div>
                <div><span class="badge green">● Course IoT server</span></div>
            </div>
        </div>
        """))
        display(HBox(
            [iot_limit_dropdown, iot_refresh_button],
            layout=Layout(align_items="center", flex_flow="row wrap", padding="10px 4px 18px 4px")
        ))
        display(HTML("""
        <div class="card">
            <h3>\U0001F321️ Sensor readings</h3>
            <p class="muted">
                Press <b>Refresh Sensor Data</b> to load the latest
                humidity and temperature values, draw the history chart,
                and save the summary to Firebase.
            </p>
        </div>
        """))


def handle_sensor_refresh(button):
    """Fetch humidity and temperature, draw charts, and save to Firebase."""
    iot_refresh_button.disabled = True
    iot_refresh_button.description = "Loading sensors..."

    try:
        build_sensor_screen()
        with main_output:
            display(HTML("""
            <div class="card">
                <h3>\U0001F4E1 Connecting to IoT server...</h3>
                <p class="muted">The first request may take a few seconds if the server is sleeping.</p>
            </div>
            """))

        limit = iot_limit_dropdown.value
        humidity_df, humidity_error = fetch_iot_feed(feed="humidity", limit=limit)
        temperature_df, temperature_error = fetch_iot_feed(feed="temperature", limit=limit)

        build_sensor_screen()
        with main_output:
            if humidity_error and temperature_error:
                display(HTML("""
                <div class="card">
                    <h2 class="section-title">❌ Sensors unavailable</h2>
                    <p class="muted">The sensor server could not be reached. Try again in a moment.</p>
                </div>
                """))
                return

            latest_humidity = extract_latest_feed_value(humidity_df)
            latest_temperature = extract_latest_feed_value(temperature_df)

            if humidity_df is not None and not humidity_df.empty:
                firebase_save_sensor_history("humidity", humidity_df)
            if temperature_df is not None and not temperature_df.empty:
                firebase_save_sensor_history("temperature", temperature_df)

            summary = {
                "air_humidity": latest_humidity,
                "temperature": latest_temperature,
                "updated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                "source": "course_iot_server"
            }
            summary_saved = firebase_save_latest_sensor_summary(summary)

            humidity_display = format_sensor_value(latest_humidity)
            temperature_display = format_sensor_value(latest_temperature)
            firebase_status = badge("Saved to Firebase", "green") if summary_saved else badge("Not saved", "orange")

            display(HTML(f"""
            <div class="card">
                <div style="display:flex; justify-content:space-between; align-items:flex-start; gap:16px; flex-wrap:wrap;">
                    <div>
                        <h2 class="section-title">\U0001F331 Current Readings</h2>
                        <p class="muted">Latest values from the course IoT server.</p>
                    </div>
                    <div>{badge("Sensor server connected", "green")} {firebase_status}</div>
                </div>
                <div class="metric-grid">
                    {metric_card("Air Humidity", humidity_display, "%")}
                    {metric_card("Temperature", temperature_display, "°C")}
                </div>
                <p class="muted">Last updated: {html.escape(summary["updated_at"])}</p>
            </div>
            """))

            display(HTML("""
            <div class="card">
                <h2 class="section-title">\U0001F4C8 Sensor History</h2>
                <p class="muted">Recent temperature and air-humidity readings.</p>
            </div>
            """))

            try:
                fig, ax = plt.subplots(figsize=(11, 4.5))

                if (humidity_df is not None and not humidity_df.empty
                        and "created_at" in humidity_df.columns and "value" in humidity_df.columns):
                    h = humidity_df.sort_values("created_at")
                    ax.plot(h["created_at"], h["value"], marker="o", label="Air Humidity (%)")

                if (temperature_df is not None and not temperature_df.empty
                        and "created_at" in temperature_df.columns and "value" in temperature_df.columns):
                    t = temperature_df.sort_values("created_at")
                    ax.plot(t["created_at"], t["value"], marker="o", label="Temperature (°C)")

                ax.set_title("Recent Sensor Readings")
                ax.set_xlabel("Time")
                ax.set_ylabel("Measured Value")
                ax.grid(True)
                ax.legend()
                plt.xticks(rotation=30)
                plt.tight_layout()
                plt.show()

            except Exception as e:
                display(HTML(f"""
                <div class="card">
                    <h3>Chart unavailable</h3>
                    <p class="muted">Could not draw the chart. Check the Colab output for details.</p>
                </div>
                """))
                print(f"Chart error: {e}")

            if humidity_error or temperature_error:
                warnings = []
                if humidity_error:
                    warnings.append("Humidity feed could not be read.")
                if temperature_error:
                    warnings.append("Temperature feed could not be read.")
                display(HTML(f"""
                <div class="card">
                    <h3>⚠️ Partial sensor data</h3>
                    <p class="muted">{'<br>'.join(warnings)}</p>
                </div>
                """))

    finally:
        iot_refresh_button.disabled = False
        iot_refresh_button.description = "\U0001F504 Refresh Sensor Data"


iot_refresh_button.on_click(handle_sensor_refresh)


# Navigation and Application Router

In [23]:
def connection_status_bar():
    llm_on = GEMINI_CLIENT is not None
    db_on = FIREBASE_READY
    llm_cls, llm_txt = ("on", "Connected") if llm_on else ("off", "Fallback mode")
    db_cls, db_txt = ("on", "Connected") if db_on else ("off", "Offline")
    return (
        '<div class="wb-status">'
        f'<span class="wb-stat {llm_cls}"><span class="dot"></span>\U0001F9E0 LLM (Gemini): {llm_txt}</span>'
        f'<span class="wb-stat {db_cls}"><span class="dot"></span>\U0001F525 Database (Firebase): {db_txt}</span>'
        '</div>'
    )


dashboard_button = widgets.Button(
    description="\U0001F4CA Dashboard",
    layout=Layout(width="100%", height="46px", margin="5px 0"),
    style=ButtonStyle(button_color="#28a96d")
)
search_nav_button = widgets.Button(
    description="\U0001F50D Search",
    layout=Layout(width="100%", height="46px", margin="5px 0"),
    style=ButtonStyle(button_color="#ffffff")
)
upload_nav_button = widgets.Button(
    description="\U0001F4F7 Upload Image",
    layout=Layout(width="100%", height="46px", margin="5px 0"),
    style=ButtonStyle(button_color="#ffffff")
)
iot_nav_button = widgets.Button(
    description="\U0001F4E1 Sensors",
    layout=Layout(width="100%", height="46px", margin="5px 0"),
    style=ButtonStyle(button_color="#ffffff")
)
chatbot_nav_button = widgets.Button(
    description=" \U0001F4AC Chatbot",
    layout=Layout(width="100%", height="46px", margin="5px 0"),
    style=ButtonStyle(button_color="#ffffff")
)
journal_nav_button = widgets.Button(
    description="\U0001F4D3 Journal",
    layout=Layout(width="100%", height="46px", margin="5px 0"),
    style=ButtonStyle(button_color="#ffffff"),
)

NAV_BUTTONS = [
    dashboard_button, chatbot_nav_button, search_nav_button,
    upload_nav_button, iot_nav_button, journal_nav_button,
]


def set_active_button(active_button):
    for btn in NAV_BUTTONS:
        btn.style.button_color = "#ffffff"
    active_button.style.button_color = "#28a96d"


def show_dashboard(button=None):
    set_active_button(dashboard_button)
    build_dashboard_screen()


def show_upload(button=None):
    set_active_button(upload_nav_button)
    build_upload_screen()


def show_sensor(button=None):
    set_active_button(iot_nav_button)
    build_sensor_screen()


def show_search(button=None):
    set_active_button(search_nav_button)
    build_search_screen()


def show_chatbot(button=None):
    set_active_button(chatbot_nav_button)
    build_chatbot_screen()




def show_journal(button=None):
    set_active_button(journal_nav_button)
    render_plant_journal()


journal_nav_button.on_click(show_journal)

dashboard_button.on_click(show_dashboard)
upload_nav_button.on_click(show_upload)
iot_nav_button.on_click(show_sensor)
search_nav_button.on_click(show_search)
chatbot_nav_button.on_click(show_chatbot)


# Run Application

In [24]:
display(HTML(f"""
<div class="wb-app">
    <div class="wb-header">
        {connection_status_bar()}
        <div class="wb-head-row">
            <div class="wb-logo">\U0001F33F</div>
            <div class="wb-head-text">
                <h1>WombatCombat Plant Care</h1>
                <p>Academic Colab system for Firebase-based Milk Thistle monitoring, academic search, and Gemini RAG.</p>
            </div>
        </div>
        <div class="wb-pills">
            <span class="wb-pill">\U0001F525 Firebase</span>
            <span class="wb-pill">\U0001F9E0 Gemini RAG</span>
            <span class="wb-pill">\U0001F4AC NLTK Chatbot</span>
            <span class="wb-pill">\U0001F50D Academic Search</span>
        </div>
    </div>
</div>
"""))

sidebar = VBox(
    [
        widgets.HTML(
            "<div style='font-weight:800;color:#0f3d2a;font-size:12px;"
            "text-transform:uppercase;letter-spacing:.07em;margin:2px 2px 8px;'>Welcome!</div>"
        ),
        dashboard_button,
        chatbot_nav_button,
        search_nav_button,
        upload_nav_button,
        iot_nav_button,
        journal_nav_button,
    ],
    layout=Layout(
        flex="0 0 210px",
        width="210px",
        min_width="180px",
        padding="14px",
        margin="0 6px 0 0"
    )
)

main_output.layout = Layout(
    flex="1 1 360px",
    width="auto",
    min_width="0",
    overflow_x="auto"
)

app_body = HBox(
    [sidebar, main_output],
    layout=Layout(
        width="100%",
        flex_flow="row wrap",
        align_items="flex-start"
    )
)

app_body.add_class("wb-canvas")
display(app_body)

show_dashboard()
